# LeetCode 329 — Longest Increasing Path in a Matrix bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:

```

Mục tiêu của bài học:

1. Làm quen với **lưới / ma trận** (grid, matrix) trong Python từ con số 0: chỉ số `(hàng, cột)`, đi **4 hướng** bằng mảng `DIRS`, kiểm tra **biên** — và cái bẫy "chỉ số âm" rất riêng của Python.
2. Nắm **đệ quy** (recursion) và **DFS** (Depth-First Search — duyệt theo chiều sâu) trên lưới; hiểu thật kỹ vì sao bài này **không cần mảng `visited`** (đường tăng nghiêm ngặt không bao giờ quay lại được ô cũ).
3. Thấy tận mắt vì sao DFS thuần (brute force) chạy theo **hàm mũ**, rồi sửa bằng **memoization** (ghi nhớ kết quả) — tự cài bằng bảng 2 chiều và bằng `functools.lru_cache`. Đây là **cách chính** `O(m·n)`.
4. Nhìn bài toán dưới dạng **đồ thị có hướng không chu trình** (DAG) và giải bằng **topological sort kiểu Kahn** — "bóc vỏ hành" từng lớp bằng hàng đợi `deque`, **số lớp = đáp án**, hoàn toàn không đệ quy.
5. Giải thêm bằng **DP sau khi sắp xếp các ô theo giá trị** — cách nghĩ "ô nhỏ trước, ô lớn sau" rất hay dùng lại.
6. Tránh các lỗi kinh điển (dùng `>=` thay vì `>`, quên kiểm tra biên dưới, bẫy `[[0]*n]*m`, cache "dính" giữa các test, tràn giới hạn đệ quy) và nhận dạng pattern để giải các bài anh em như *Number of Increasing Paths in a Grid*, *Course Schedule*.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một ma trận số nguyên `matrix` kích thước `m x n`, hãy trả về **độ dài của đường đi tăng dài nhất** (longest increasing path) trong `matrix`.
>
> Từ mỗi ô, bạn có thể di chuyển theo **4 hướng**: trái, phải, lên hoặc xuống. Bạn **không được** đi chéo và **không được** đi ra ngoài biên (tức là không được "quấn vòng" — wrap-around — từ mép này sang mép đối diện).

**Example 1**

```text
Input: matrix = [[9,9,4],[6,6,8],[2,1,1]]
Output: 4
Giải thích: Đường đi tăng dài nhất là [1, 2, 6, 9].
```

**Example 2**

```text
Input: matrix = [[3,4,5],[3,2,6],[2,2,1]]
Output: 4
Giải thích: Đường đi tăng dài nhất là [3, 4, 5, 6]. Không được đi chéo.
```

**Example 3**

```text
Input: matrix = [[1]]
Output: 1
```

**Constraints**

- `m == matrix.length`
- `n == matrix[i].length`
- `1 <= m, n <= 200`
- `0 <= matrix[i][j] <= 2^31 - 1`

### Ký hiệu dùng suốt notebook

- `m` = số hàng, `n` = số cột, `N = m · n` = tổng số ô (tối đa `200 · 200 = 40 000`).
- Ô ở hàng `r`, cột `c` viết là `(r, c)`; giá trị của nó là `matrix[r][c]`. Ví dụ `(2,1)=1` nghĩa là "ô hàng 2, cột 1, có giá trị 1".
- "Độ dài" của đường đi = **số ô** trên đường đi (không phải số bước). Một ô đứng một mình là đường đi độ dài `1`.

### 1.1 Hiểu đề bằng hình

Vẽ Example 1, đánh dấu đường đi đáp án `1 → 2 → 6 → 9` bằng dấu `*`:

```text
         c=0   c=1   c=2
r=0  [  *9     9     4  ]
r=1  [  *6     6     8  ]
r=2  [  *2    *1     1  ]
```

Đi từ ô `(2,1)=1` sang **trái** tới `(2,0)=2`, rồi **lên** `(1,0)=6`, rồi **lên** `(0,0)=9`. Mỗi bước sang ô kề cạnh và giá trị **tăng ngặt**: `1 < 2 < 6 < 9`.

| Dãy ô | Hợp lệ? | Lý do |
|---|:---:|---|
| `(2,1)=1 → (2,0)=2 → (1,0)=6 → (0,0)=9` | ✅ | kề cạnh, tăng ngặt, độ dài 4 |
| `(2,1)=1 → (2,0)=2 → (1,0)=6 → (1,1)=6 → (1,2)=8` | ❌ | `6 → 6` **không tăng** (bằng nhau không được) |
| `(2,2)=1 → (1,1)=6` | ❌ | hai ô nằm **chéo** nhau |
| `(0,2)=4 → (0,0)=9` | ❌ | `(0,2)` và `(0,0)` không kề nhau — đi "quấn vòng" qua mép là cấm |
| `(1,2)=8` | ✅ | một ô đơn lẻ cũng là đường đi, độ dài 1 |

Vài suy luận nhỏ mà rất quan trọng:

- **"Tăng" ở đây là tăng nghiêm ngặt** (strictly increasing, `<` chứ không phải `<=`). Bằng chứng ngay trong Example 1: nếu cho phép bằng nhau thì `1 → 2 → 6 → 6 → 8` dài 5, nhưng đáp án đề là 4.
- **Example 2 nhắc "không được đi chéo"** vì nếu được đi chéo thì có đường `(2,2)=1 ↘ (1,1)=2 ↘ (0,0)=3 → 4 → 5 → 6` dài 6.
- Đường đi có thể **bắt đầu ở bất kỳ ô nào** và đi theo **bất kỳ hướng nào** (không bắt buộc từ góc trên trái), nên đáp án luôn `>= 1`.
- Giá trị lên tới `2^31 - 1` nhưng ta chỉ **so sánh** `<`, `>`, không cộng trừ giá trị, nên không lo tràn số.

Cell dưới định nghĩa 3 ví dụ của đề, một hàm in lưới cho dễ nhìn, và một hàm **kiểm tra một dãy ô có phải đường tăng hợp lệ không** — dùng nó để xác nhận bảng trên.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache
import math
import random
import sys
import time

ex1 = [[9, 9, 4], [6, 6, 8], [2, 1, 1]]
ex2 = [[3, 4, 5], [3, 2, 6], [2, 2, 1]]
ex3 = [[1]]


def print_grid(grid, title=None, width=3):
    # In một lưới 2 chiều, mỗi hàng một dòng, các số căn phải cho thẳng cột.
    if title:
        print(title)
    for row in grid:
        print(" ".join(f"{x:>{width}}" for x in row))


def is_increasing_path(matrix, cells):
    # cells: danh sách (r, c). Kiểm tra: trong biên, kề cạnh, giá trị tăng ngặt.
    m, n = len(matrix), len(matrix[0])
    for r, c in cells:
        if not (0 <= r < m and 0 <= c < n):
            return False
    for (r1, c1), (r2, c2) in zip(cells, cells[1:]):
        if abs(r1 - r2) + abs(c1 - c2) != 1:          # phải kề cạnh (không chéo, không nhảy)
            return False
        if not matrix[r2][c2] > matrix[r1][c1]:       # phải tăng ngặt
            return False
    return True


print_grid(ex1, "Example 1:")
print("đáp án 1-2-6-9   :", is_increasing_path(ex1, [(2, 1), (2, 0), (1, 0), (0, 0)]))
print("có bước 6 -> 6   :", is_increasing_path(ex1, [(2, 1), (2, 0), (1, 0), (1, 1), (1, 2)]))
print("đi chéo          :", is_increasing_path(ex1, [(2, 2), (1, 1)]))
print("quấn vòng qua mép:", is_increasing_path(ex1, [(0, 2), (0, 0)]))
print("một ô đơn lẻ     :", is_increasing_path(ex1, [(1, 2)]))

Example 1:
  9   9   4
  6   6   8
  2   1   1
đáp án 1-2-6-9   : True
có bước 6 -> 6   : False
đi chéo          : False
quấn vòng qua mép: False
một ô đơn lẻ     : True


## 2. Nền tảng: lưới (ma trận) trong Python

### 2.1 Ma trận là "list của các list"

Python không có kiểu "ma trận" riêng; ta dùng **list chứa các list**. Mỗi list con là **một hàng**.

```text
          c=0  c=1  c=2
r=0   [    9,   9,   4 ]   <- matrix[0]
r=1   [    6,   6,   8 ]   <- matrix[1]
r=2   [    2,   1,   1 ]   <- matrix[2]
```

- `matrix[r]` là cả hàng `r` (một list).
- `matrix[r][c]` là ô ở hàng `r`, cột `c`: **chỉ số hàng trước, chỉ số cột sau**.
- `m = len(matrix)` là số hàng; `n = len(matrix[0])` là số cột (độ dài của một hàng bất kỳ).
- `r` tăng nghĩa là đi **xuống**, `c` tăng nghĩa là đi sang **phải**.

Duyệt toàn bộ ô bằng hai vòng `for` lồng nhau: vòng ngoài chạy hàng, vòng trong chạy cột.

In [2]:
matrix = ex1
m = len(matrix)          # số hàng
n = len(matrix[0])       # số cột
print("m =", m, ", n =", n)
print("matrix[2]    =", matrix[2])       # cả hàng 2
print("matrix[2][1] =", matrix[2][1])    # hàng 2, cột 1
print("matrix[0][2] =", matrix[0][2])    # hàng 0, cột 2

print("Duyệt mọi ô theo thứ tự hàng rồi cột:")
for r in range(m):
    print("  ", [(r, c, matrix[r][c]) for c in range(n)])

m = 3 , n = 3
matrix[2]    = [2, 1, 1]
matrix[2][1] = 1
matrix[0][2] = 4
Duyệt mọi ô theo thứ tự hàng rồi cột:
   [(0, 0, 9), (0, 1, 9), (0, 2, 4)]
   [(1, 0, 6), (1, 1, 6), (1, 2, 8)]
   [(2, 0, 2), (2, 1, 1), (2, 2, 1)]


### 2.2 Bốn hướng đi, kiểm tra biên và cái bẫy "chỉ số âm"

Từ ô `(r, c)`, bốn ô kề cạnh là:

| Hướng | Đổi hàng `dr` | Đổi cột `dc` | Ô mới |
|---|:---:|:---:|---|
| lên | `-1` | `0` | `(r-1, c)` |
| xuống | `+1` | `0` | `(r+1, c)` |
| trái | `0` | `-1` | `(r, c-1)` |
| phải | `0` | `+1` | `(r, c+1)` |

Thay vì viết 4 khối `if` gần giống nhau, ta gom 4 cặp `(dr, dc)` vào một list `DIRS` rồi lặp:

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc                  # "new row", "new column"
```

Ô mới chỉ hợp lệ khi nằm **trong lưới**: `0 <= nr < m and 0 <= nc < n`. Python cho phép viết so sánh "kẹp hai đầu" `0 <= nr < m` — nghĩa là `0 <= nr and nr < m`.

**Bẫy riêng của Python**: chỉ số âm **không báo lỗi**! `matrix[-1]` là hàng **cuối cùng**, `row[-1]` là phần tử **cuối**. Nếu quên vế `0 <= nr`, khi đứng ở hàng 0 mà đi "lên", code sẽ lặng lẽ đọc hàng cuối — chính là "quấn vòng" mà đề cấm. Ta sẽ thấy lỗi này làm sai đáp án ở mục 13.

Cuối cell dưới, ta in luôn các **"mũi tên"**: từ mỗi ô, những hàng xóm nào **lớn hơn** nó — đó chính là những nước đi hợp lệ của một đường tăng.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải


def neighbors(matrix, r, c):
    # Trả về các ô kề cạnh (r, c) nằm TRONG lưới, theo thứ tự lên, xuống, trái, phải.
    m, n = len(matrix), len(matrix[0])
    result = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            result.append((nr, nc))
    return result


print("hàng xóm của ô giữa (1,1):", neighbors(ex1, 1, 1))
print("hàng xóm của góc (0,0)   :", neighbors(ex1, 0, 0))
print("bẫy chỉ số âm: ex1[-1][0] =", ex1[-1][0], "-> đó là ô (2,0), KHÔNG phải ô 'phía trên' (0,0)")

print("\nMũi tên từ mỗi ô sang hàng xóm LỚN HƠN (Example 1):")
for r in range(3):
    for c in range(3):
        bigger = [f"({nr},{nc})={ex1[nr][nc]}" for nr, nc in neighbors(ex1, r, c) if ex1[nr][nc] > ex1[r][c]]
        print(f"  ({r},{c})={ex1[r][c]} -> {', '.join(bigger) if bigger else 'không có (đứng ở đỉnh)'}")

hàng xóm của ô giữa (1,1): [(0, 1), (2, 1), (1, 0), (1, 2)]
hàng xóm của góc (0,0)   : [(1, 0), (0, 1)]
bẫy chỉ số âm: ex1[-1][0] = 2 -> đó là ô (2,0), KHÔNG phải ô 'phía trên' (0,0)

Mũi tên từ mỗi ô sang hàng xóm LỚN HƠN (Example 1):
  (0,0)=9 -> không có (đứng ở đỉnh)
  (0,1)=9 -> không có (đứng ở đỉnh)
  (0,2)=4 -> (1,2)=8, (0,1)=9
  (1,0)=6 -> (0,0)=9
  (1,1)=6 -> (0,1)=9, (1,2)=8
  (1,2)=8 -> không có (đứng ở đỉnh)
  (2,0)=2 -> (1,0)=6
  (2,1)=1 -> (1,1)=6, (2,0)=2
  (2,2)=1 -> (1,2)=8


### 2.3 Đệ quy: khởi động bằng phiên bản **một chiều** của bài toán

**Đệ quy** (recursion) là khi một hàm **gọi lại chính nó** cho một bài toán **nhỏ hơn**. Hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp cơ sở): đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Recursive case**: gọi chính hàm đó cho bài toán con, rồi **kết hợp** kết quả.

Khởi động bằng bài toán "một chiều" giống hệt bài chính: cho một hàng `row`, đứng ở vị trí `i` và **chỉ được đi sang phải**, đường tăng dài nhất bắt đầu tại `i` dài bao nhiêu?

```text
run_right(i) = 1                        nếu i là ô cuối hoặc row[i+1] <= row[i]   (base case)
run_right(i) = 1 + run_right(i + 1)     nếu row[i+1] > row[i]                      (recursive case)
```

Với `row = [1, 3, 5, 4, 6]`: `run_right(2) = 1` (vì `4 <= 5`), `run_right(1) = 1 + 1 = 2`, `run_right(0) = 1 + 2 = 3` → đường `1, 3, 5`.

Khi `run_right(0)` gọi `run_right(1)`, nó **tạm dừng và chờ**. Các lời gọi đang chờ xếp chồng lên nhau trong **call stack** (ngăn xếp lời gọi): vào sau thì xong trước. Cell dưới in "vào"/"ra" thụt lề theo độ sâu để thấy rõ.

Bài 329 chỉ khác ở chỗ: thay vì 1 hướng ta có **4 hướng**, và ta lấy **`max`** của các hướng đi được.

In [4]:
def run_right(row, i, depth=0):
    pad = "    " * depth
    print(f"{pad}vào run_right({i}), row[{i}] = {row[i]}")
    if i + 1 < len(row) and row[i + 1] > row[i]:     # đi tiếp sang phải được
        result = 1 + run_right(row, i + 1, depth + 1)
    else:                                            # base case: dừng tại đây
        result = 1
    print(f"{pad}ra  run_right({i}) = {result}")
    return result


row = [1, 3, 5, 4, 6]
print("Kết quả:", run_right(row, 0))

vào run_right(0), row[0] = 1
    vào run_right(1), row[1] = 3
        vào run_right(2), row[2] = 5
        ra  run_right(2) = 1
    ra  run_right(1) = 2
ra  run_right(0) = 3
Kết quả: 3


### 2.4 DFS trên lưới: liệt kê **mọi** đường tăng xuất phát từ một ô

**DFS** (Depth-First Search — duyệt theo chiều sâu) = cứ đi sâu nhất có thể theo một hướng, hết đường mới **quay lui** (backtrack) để thử hướng khác. Đệ quy chính là cách tự nhiên để viết DFS.

Để "nhìn thấy" các đường đi, ta giữ một list `path` chứa các ô đang đi:

- Trước khi bước sang ô mới: `path.append(ô mới)`.
- Sau khi lời gọi đệ quy cho ô đó trả về: `path.pop()` — gỡ ô đó ra để thử hướng khác.
- Khi đứng ở ô **không còn hàng xóm nào lớn hơn**, đường không kéo dài thêm được nữa → ghi lại một bản sao `list(path)`.

Ta gọi các đường như vậy là đường **tối đại** (maximal — không nối dài thêm được). Đường dài nhất chắc chắn là một đường tối đại.

Thử với ô `(2,1)=1` của Example 1. Dự đoán trước bằng bảng mũi tên ở mục 2.2: từ `1` đi lên `6` (rồi lên `9` hoặc sang phải `8`), hoặc sang trái `2` → `6` → `9`. Vậy có 3 đường tối đại.

In [5]:
def all_maximal_paths(matrix, r0, c0):
    paths = []
    path = [(r0, c0)]

    def dfs(r, c):
        extended = False
        for nr, nc in neighbors(matrix, r, c):
            if matrix[nr][nc] > matrix[r][c]:        # chỉ bước sang ô LỚN HƠN
                extended = True
                path.append((nr, nc))
                dfs(nr, nc)
                path.pop()                           # quay lui
        if not extended:                             # hết đường -> ghi lại
            paths.append(list(path))

    dfs(r0, c0)
    return paths


for p in all_maximal_paths(ex1, 2, 1):
    print(f"độ dài {len(p)}: ô {p}  giá trị {[ex1[r][c] for r, c in p]}")

độ dài 3: ô [(2, 1), (1, 1), (0, 1)]  giá trị [1, 6, 9]
độ dài 3: ô [(2, 1), (1, 1), (1, 2)]  giá trị [1, 6, 8]
độ dài 4: ô [(2, 1), (2, 0), (1, 0), (0, 0)]  giá trị [1, 2, 6, 9]


### 2.5 Vì sao bài này **không cần** mảng `visited`?

Ở các bài lưới quen thuộc (ví dụ *Number of Islands*), DFS phải đánh dấu `visited` (đã thăm), nếu không sẽ đi qua lại giữa hai ô kề nhau mãi mãi.

Ở bài này thì **không cần**, vì mỗi bước đi giá trị **tăng ngặt**:

```text
matrix[ô 1] < matrix[ô 2] < matrix[ô 3] < ... < matrix[ô k]
```

Giả sử một đường đi quay lại một ô đã đi qua, ví dụ ô `X` xuất hiện ở vị trí `i` và `j` (`i < j`). Khi đó giá trị tại vị trí `j` phải **lớn hơn hẳn** giá trị tại vị trí `i` — nhưng đó lại cùng là `matrix[X]`. Vô lý: không số nào lớn hơn chính nó. Vậy **mọi đường tăng tự động không lặp ô**, chính điều kiện `>` đã làm thay việc của `visited`.

Hệ quả còn quan trọng hơn (sẽ dùng ở mục 3.4): vì không có `visited`, **câu trả lời tại một ô chỉ phụ thuộc vào chính ô đó**, không phụ thuộc vào việc ta đã đi tới nó bằng đường nào. Đây là điều kiện để **ghi nhớ** (memo) được.

Cell dưới kiểm chứng bằng máy: liệt kê mọi đường tối đại từ **mọi** ô của Example 1 và 2, xác nhận không đường nào lặp ô, và tìm đường dài nhất.

In [6]:
for name, mat in [("Example 1", ex1), ("Example 2", ex2)]:
    total, longest = 0, []
    for r in range(len(mat)):
        for c in range(len(mat[0])):
            for p in all_maximal_paths(mat, r, c):
                assert len(set(p)) == len(p), "có ô bị lặp!"     # set bỏ trùng -> nếu lặp thì ngắn hơn
                assert is_increasing_path(mat, p)
                total += 1
                if len(p) > len(longest):
                    longest = p
    print(f"{name}: {total} đường tối đại, không đường nào lặp ô; dài nhất = {len(longest)}:",
          [mat[r][c] for r, c in longest])

Example 1: 13 đường tối đại, không đường nào lặp ô; dài nhất = 4: [1, 2, 6, 9]
Example 2: 12 đường tối đại, không đường nào lặp ô; dài nhất = 4: [3, 4, 5, 6]


## 3. Cách 1 — DFS thuần (brute force)

### 3.1 Ý tưởng tự nhiên nhất

Đặt tên cho câu hỏi con:

> `longest_from(r, c)` = độ dài đường tăng dài nhất **bắt đầu tại** ô `(r, c)`.

Một đường tăng bắt đầu tại `(r, c)` chỉ có hai khả năng:

- dừng ngay tại `(r, c)` → độ dài `1`;
- bước sang một hàng xóm **lớn hơn** `(nr, nc)`, rồi tiếp tục là một đường tăng bắt đầu tại `(nr, nc)` → độ dài `1 + longest_from(nr, nc)`.

Nên:

```text
longest_from(r, c) = 1 + max( longest_from(nr, nc) với mọi hàng xóm (nr, nc) lớn hơn )
                   = 1      nếu không có hàng xóm nào lớn hơn       (base case)
```

Đáp án của bài = `max(longest_from(r, c))` trên **mọi** ô.

Tính tay vài ô của Example 1 (dùng bảng mũi tên ở mục 2.2):

- `(0,0)=9` không có ô lớn hơn → `1`.
- `(1,0)=6` chỉ đi lên `(0,0)=9` → `1 + 1 = 2`.
- `(2,0)=2` chỉ đi lên `(1,0)=6` → `1 + 2 = 3`.
- `(2,1)=1` đi lên `(1,1)=6` (được 2) hoặc sang trái `(2,0)=2` (được 3) → `1 + max(2, 3) = 4`. Đó là đáp án.

Cú pháp mới: `max(longest_from(r, c) for r in range(m) for c in range(n))` là **generator expression** có hai vòng `for` — đọc từ trái sang phải giống hệt hai vòng lồng nhau: vòng ngoài `r`, vòng trong `c`.

In [7]:
def lip_bruteforce(matrix):
    m, n = len(matrix), len(matrix[0])

    def longest_from(r, c):
        best = 1                                   # dừng ngay tại (r, c)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + longest_from(nr, nc))
        return best

    return max(longest_from(r, c) for r in range(m) for c in range(n))


print("hai vòng for trong một dòng:", [(r, c) for r in range(2) for c in range(3)])
print("Example 1:", lip_bruteforce(ex1))   # 4
print("Example 2:", lip_bruteforce(ex2))   # 4
print("Example 3:", lip_bruteforce(ex3))   # 1

hai vòng for trong một dòng: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
Example 1: 4
Example 2: 4
Example 3: 1


### 3.2 Brute force tốn bao nhiêu lời gọi?

Đúng là ra đáp án, nhưng hãy **đếm** xem mỗi ô bị gọi `longest_from` bao nhiêu lần. Ở Example 1, ô `(1,2)=8` được hỏi từ chính nó (vòng ngoài), từ `(0,2)=4`, từ `(1,1)=6` (hai lần: một lần khi vòng ngoài tới `(1,1)`, một lần khi đi `(2,1) → (1,1)`), và từ `(2,2)=1` → **5 lần**, lần nào cũng tính lại từ đầu cùng một kết quả `1`.

Với lưới 3×3 thì không sao, nhưng hãy thử lưới `k × k` có `matrix[i][j] = i + j`:

```text
k = 4:   0 1 2 3
         1 2 3 4
         2 3 4 5
         3 4 5 6
```

Từ mỗi ô, cả ô **bên phải** lẫn ô **bên dưới** đều lớn hơn, nên số đường tăng từ góc `(0,0)` tới góc `(k-1,k-1)` bằng số cách sắp xếp `k-1` bước "xuống" và `k-1` bước "phải" = $\binom{2k-2}{k-1}$ — tăng theo **hàm mũ**. Brute force đi hết từng đường một.

Cell dưới in số lần gọi của từng ô ở Example 1, rồi đếm tổng số lời gọi trên lưới `i + j` với `k = 2..10`.

In [8]:
def lip_bruteforce_count(matrix):
    # Giống lip_bruteforce, nhưng đếm số lần mỗi ô bị gọi.
    m, n = len(matrix), len(matrix[0])
    calls = [[0] * n for _ in range(m)]

    def longest_from(r, c):
        calls[r][c] += 1
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + longest_from(nr, nc))
        return best

    ans = max(longest_from(r, c) for r in range(m) for c in range(n))
    return ans, calls


ans, calls = lip_bruteforce_count(ex1)
print_grid(calls, "Số lần mỗi ô bị gọi (Example 1):")
print("tổng:", sum(map(sum, calls)), "lời gọi cho 9 ô\n")

brute_calls = {}
print(" k | đáp án | tổng lời gọi | thời gian")
for k in range(2, 11):
    grid = [[i + j for j in range(k)] for i in range(k)]
    t0 = time.perf_counter()
    ans, calls = lip_bruteforce_count(grid)
    brute_calls[k] = sum(map(sum, calls))
    print(f"{k:2} | {ans:6} | {brute_calls[k]:12,} | {(time.perf_counter() - t0) * 1000:7.1f} ms")

print("\nSố đường tăng từ góc tới góc của lưới 200x200 kiểu i+j có",
      len(str(math.comb(398, 199))), "chữ số!")

Số lần mỗi ô bị gọi (Example 1):
  4   4   1
  3   2   5
  2   1   1
tổng: 23 lời gọi cho 9 ô

 k | đáp án | tổng lời gọi | thời gian
 2 |      3 |           10 |     0.0 ms
 3 |      5 |           53 |     0.0 ms
 4 |      7 |          226 |     0.1 ms
 5 |      9 |          887 |     0.4 ms
 6 |     11 |        3,382 |     1.7 ms
 7 |     13 |       12,805 |     6.4 ms
 8 |     15 |       48,538 |    24.0 ms
 9 |     17 |      184,655 |    89.6 ms


10 |     19 |      705,310 |   358.3 ms

Số đường tăng từ góc tới góc của lưới 200x200 kiểu i+j có 119 chữ số!


### 3.3 Brute force vướng ở đâu? → Mấu chốt cải tiến

Nhìn bảng trên: mỗi khi `k` tăng 1, số lời gọi tăng **gần gấp 4**. Với `k = 200` (giới hạn đề), con số có hơn một trăm chữ số — không máy nào chạy nổi.

Nhưng để ý: lưới `10 × 10` chỉ có **100 ô**, mà ta gọi hàm hơn **700 nghìn** lần. Nghĩa là gần như toàn bộ công việc là **tính lại những câu hỏi đã có đáp án**. Đây gọi là **bài toán con chồng lặp** (overlapping subproblems).

Mấu chốt (đã chuẩn bị ở mục 2.5):

> `longest_from(r, c)` **chỉ phụ thuộc vào ô `(r, c)`** — không phụ thuộc ta đến ô đó bằng đường nào, vì không có `visited` nào cả.

Vậy chỉ cần tính mỗi ô **một lần**, **ghi nhớ** kết quả, lần sau hỏi lại thì trả luôn. Kỹ thuật này gọi là **memoization**. Trước khi áp vào lưới, ta học nó trên ví dụ kinh điển.

> Ghi chú cho người tò mò: nếu đề cho phép đi qua các ô **bằng nhau** hoặc **nhỏ hơn** (đường đi "đơn" bất kỳ), ta buộc phải mang theo `visited`, câu trả lời khi đó phụ thuộc vào cả tập ô đã đi → không memo được. Bài "đường đi đơn dài nhất" trên đồ thị tổng quát là bài cực khó (NP-hard). Chính điều kiện **tăng ngặt** đã biến bài 329 thành bài dễ chịu.

## 4. Nền tảng: Memoization (ghi nhớ kết quả)

### 4.1 Ví dụ kinh điển: Fibonacci

Dãy Fibonacci: `fib(0) = 0`, `fib(1) = 1`, `fib(k) = fib(k-1) + fib(k-2)`.

Viết đệ quy thẳng thì `fib(5)` gọi `fib(4)` và `fib(3)`; `fib(4)` lại gọi `fib(3)` lần nữa... cây lời gọi phình theo hàm mũ — đúng căn bệnh của Cách 1.

**Memoization** chữa bằng một cuốn "sổ ghi chép" `memo` (thường là `dict`):

1. Vào hàm: nếu `k` đã có trong `memo` → trả luôn `memo[k]`.
2. Nếu chưa: tính như bình thường, **ghi `memo[k] = kết quả`** rồi mới trả về.

Python còn có sẵn "sổ ghi chép tự động": decorator `@lru_cache(maxsize=None)` trong module `functools`.

- **Decorator** là dòng `@tên` đặt ngay trên `def`: nó "bọc" hàm lại để thêm tính năng. Ở đây tính năng là: tự nhớ kết quả theo **bộ tham số** (các tham số phải hashable — số, chuỗi, tuple... đều được; list thì không).
- `maxsize=None` nghĩa là nhớ không giới hạn. (`functools.cache` từ Python 3.9 cũng chính là `lru_cache(maxsize=None)`.)
- `hàm.cache_info()` cho biết số lần trúng cache (`hits`) và trượt (`misses`); `hàm.cache_clear()` xóa sổ.

In [9]:
naive_calls = 0


def fib_naive(k):
    global naive_calls
    naive_calls += 1
    if k < 2:
        return k
    return fib_naive(k - 1) + fib_naive(k - 2)


memo_calls = 0
memo = {}


def fib_memo(k):
    global memo_calls
    memo_calls += 1
    if k in memo:                 # 1. đã có trong sổ -> trả luôn
        return memo[k]
    result = k if k < 2 else fib_memo(k - 1) + fib_memo(k - 2)
    memo[k] = result              # 2. ghi vào sổ rồi mới trả
    return result


print("fib_naive(25) =", fib_naive(25), "| số lời gọi:", f"{naive_calls:,}")
print("fib_memo(25)  =", fib_memo(25), "| số lời gọi:", memo_calls)


@lru_cache(maxsize=None)
def fib_cached(k):
    return k if k < 2 else fib_cached(k - 1) + fib_cached(k - 2)


print("fib_cached(80) =", fib_cached(80))
print(fib_cached.cache_info())
fib_cached.cache_clear()
print("sau cache_clear:", fib_cached.cache_info())

fib_naive(25) = 75025 | số lời gọi: 242,785
fib_memo(25)  = 75025 | số lời gọi: 49
fib_cached(80) = 23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)
sau cache_clear: CacheInfo(hits=0, misses=0, maxsize=None, currsize=0)


### 4.2 Memo bằng **bảng 2 chiều** — và cái bẫy `[[0] * n] * m`

Ở bài lưới, trạng thái là cặp `(r, c)`, nên thay cho `dict` ta có thể dùng một **bảng** `memo` cùng kích thước `m × n`:

- `memo[r][c] == 0` nghĩa là **chưa tính**. Dùng `0` làm "cờ chưa tính" được vì đáp án thật của mọi ô luôn `>= 1`.
- `memo[r][c] = k > 0` nghĩa là đã biết đường tăng dài nhất từ `(r, c)` là `k`.

Truy cập bảng nhanh hơn `dict` một chút và in ra rất dễ nhìn.

**Bẫy kinh điển khi tạo bảng**: `[[0] * n] * m` **không** tạo `m` hàng độc lập. Nó tạo **một** hàng rồi lặp lại **cùng một đối tượng** `m` lần (aliasing — nhiều tên cùng trỏ một vật). Sửa một ô ở hàng 0 thì "mọi hàng" cùng đổi. Cách đúng là **list comprehension**: `[[0] * n for _ in range(m)]` — mỗi vòng tạo một hàng mới. (`_` là tên biến quy ước cho "biến không dùng tới".)

Toán tử `is` kiểm tra hai tên có trỏ **cùng một đối tượng** không.

In [10]:
bad = [[0] * 3] * 2                     # SAI: 2 "hàng" là cùng một list
good = [[0] * 3 for _ in range(2)]      # ĐÚNG: mỗi hàng là một list mới

bad[0][0] = 7
good[0][0] = 7
print("bad :", bad)                     # [[7, 0, 0], [7, 0, 0]]  <- hàng 1 cũng đổi!
print("good:", good)                    # [[7, 0, 0], [0, 0, 0]]
print("bad[0] is bad[1]  ->", bad[0] is bad[1])
print("good[0] is good[1] ->", good[0] is good[1])

bad : [[7, 0, 0], [7, 0, 0]]
good: [[7, 0, 0], [0, 0, 0]]
bad[0] is bad[1]  -> True
good[0] is good[1] -> False


## 5. Cách 2 (cách chính) — DFS + memo, `O(m · n)`

### 5.1 Ghép lại

Lấy nguyên hàm `longest_from` của Cách 1 (đổi tên thành `dfs`) và thêm đúng **hai dòng memo**:

```python
def dfs(r, c):
    if memo[r][c] != 0:          # (1) đã tính rồi -> dùng lại
        return memo[r][c]
    best = 1
    for 4 hướng:
        nếu hàng xóm trong lưới và LỚN HƠN:
            best = max(best, 1 + dfs(hàng xóm))
    memo[r][c] = best            # (2) ghi sổ trước khi trả về
    return best
```

Vòng ngoài vẫn gọi `dfs(r, c)` cho mọi ô và lấy `max`. Ô nào đã được tính "ké" trong lúc DFS từ ô khác thì vòng ngoài chỉ tra sổ, tốn `O(1)`.

Vì sao **thứ tự tính tự động đúng**? `dfs(r, c)` cần kết quả của các ô **lớn hơn**; đệ quy sẽ tự đi tính những ô đó trước. Và vì giá trị tăng ngặt dọc đường đệ quy, chuỗi lời gọi luôn dừng (không thể vòng tròn).

Ta viết hai phiên bản giống hệt nhau về ý tưởng:

- `lip_memo`: memo bằng **bảng** (tự cài, dễ in ra để quan sát).
- `lip_memo_cache`: memo bằng **`@lru_cache`** (ngắn hơn). Hàm `dfs` được định nghĩa **bên trong** để mỗi lần gọi `lip_memo_cache` có một cache mới tinh.

In [11]:
def lip_memo(matrix, return_memo=False):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]          # 0 = chưa tính (đáp án thật luôn >= 1)

    def dfs(r, c):
        if memo[r][c] != 0:                     # (1) đã có trong sổ
            return memo[r][c]
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best                       # (2) ghi sổ
        return best

    ans = max(dfs(r, c) for r in range(m) for c in range(n))
    return (ans, memo) if return_memo else ans


def lip_memo_cache(matrix):
    m, n = len(matrix), len(matrix[0])

    @lru_cache(maxsize=None)
    def dfs(r, c):
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n))


for name, mat in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3)]:
    ans, memo_grid = lip_memo(mat, return_memo=True)
    print(f"{name}: lip_memo = {ans}, lip_memo_cache = {lip_memo_cache(mat)}")
    print_grid(memo_grid, "  bảng memo (đường tăng dài nhất BẮT ĐẦU tại mỗi ô):")

Example 1: lip_memo = 4, lip_memo_cache = 4
  bảng memo (đường tăng dài nhất BẮT ĐẦU tại mỗi ô):
  1   1   2
  2   2   1
  3   4   2
Example 2: lip_memo = 4, lip_memo_cache = 4
  bảng memo (đường tăng dài nhất BẮT ĐẦU tại mỗi ô):
  4   3   2
  1   4   1
  2   1   2
Example 3: lip_memo = 1, lip_memo_cache = 1
  bảng memo (đường tăng dài nhất BẮT ĐẦU tại mỗi ô):
  1


### 5.2 Mỗi ô chỉ "tính thật" đúng một lần

Đếm lại trên lưới `i + j` như mục 3.2. Có hai loại lời gọi `dfs`:

- **tính thật**: lần đầu gặp ô, phải chạy vòng 4 hướng — xảy ra đúng **1 lần mỗi ô**, tức `k²` lần;
- **tra sổ**: ô đã có memo, trả ngay — mỗi ô bị tra tối đa bằng số hàng xóm **nhỏ hơn** nó (≤ 4) cộng 1 lần từ vòng ngoài.

Vậy tổng số lời gọi `<= N + 4N`, tức là `O(N)` với `N = m · n`. So với brute force:

In [12]:
def lip_memo_count(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]
    stats = {"total": 0, "computed": 0}

    def dfs(r, c):
        stats["total"] += 1
        if memo[r][c]:
            return memo[r][c]
        stats["computed"] += 1
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best
        return best

    ans = max(dfs(r, c) for r in range(m) for c in range(n))
    return ans, stats


print(" k | số ô | brute force: lời gọi | memo: tính thật | memo: tổng lời gọi")
for k in range(2, 11):
    grid = [[i + j for j in range(k)] for i in range(k)]
    ans, stats = lip_memo_count(grid)
    print(f"{k:2} | {k * k:4} | {brute_calls[k]:20,} | {stats['computed']:15} | {stats['total']:18}")

 k | số ô | brute force: lời gọi | memo: tính thật | memo: tổng lời gọi
 2 |    4 |                   10 |               4 |                  8
 3 |    9 |                   53 |               9 |                 21
 4 |   16 |                  226 |              16 |                 40
 5 |   25 |                  887 |              25 |                 65
 6 |   36 |                3,382 |              36 |                 96
 7 |   49 |               12,805 |              49 |                133
 8 |   64 |               48,538 |              64 |                176
 9 |   81 |              184,655 |              81 |                225
10 |  100 |              705,310 |             100 |                280


## 6. Trace từng bước DFS + memo trên Example 1

Hàm `trace_lip_memo` in ra:

- `[vòng ngoài]` khi vòng `for r, c` bên ngoài bắt đầu hỏi một ô;
- `vào (r,c)=v` khi một ô được **tính thật** lần đầu; thụt lề = độ sâu đệ quy;
- `... đã có memo = k` khi ô đó **tra sổ**;
- `xong (r,c)=v: memo = k` khi ô tính xong và được ghi sổ, kèm hàng xóm đã cho kết quả tốt nhất.

Dự đoán trước bằng tay (thứ tự 4 hướng: lên, xuống, trái, phải). Đọc cột "Thứ tự xong" để thấy ô nào được ghi sổ trước:

| Thứ tự xong | Ô | Giá trị | Hàng xóm lớn hơn → memo của chúng | memo = 1 + max |
|:---:|:---:|:---:|---|:---:|
| 1 | (0,0) | 9 | không có | **1** |
| 2 | (0,1) | 9 | không có | **1** |
| 3 | (1,2) | 8 | không có (được tính "ké" bên trong (0,2)) | **1** |
| 4 | (0,2) | 4 | (1,2)=8 → 1, (0,1)=9 → 1 (tra sổ) | **2** |
| 5 | (1,0) | 6 | (0,0)=9 → 1 (tra sổ) | **2** |
| 6 | (1,1) | 6 | (0,1)=9 → 1, (1,2)=8 → 1 (đều tra sổ) | **2** |
| 7 | (2,0) | 2 | (1,0)=6 → 2 (tra sổ) | **3** |
| 8 | (2,1) | 1 | (1,1)=6 → 2, (2,0)=2 → 3 (đều tra sổ) | **4** |
| 9 | (2,2) | 1 | (1,2)=8 → 1 (tra sổ) | **2** |

Để ý: khi vòng ngoài tới `(1,2)`, ô này **đã có memo** (tính ké từ bước 3) nên chỉ tra sổ, không tính lại. Và các ô ở hàng dưới cùng — những ô "nhỏ" — đều được tính rất nhanh vì mọi ô lớn hơn chúng đã nằm sẵn trong sổ.

In [13]:
def trace_lip_memo(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]

    def dfs(r, c, depth):
        pad = "[vòng ngoài] " if depth == 0 else "    " * depth
        if memo[r][c]:
            print(f"{pad}({r},{c})={matrix[r][c]}: đã có memo = {memo[r][c]} -> dùng lại")
            return memo[r][c]
        print(f"{pad}vào ({r},{c})={matrix[r][c]}")
        best, via = 1, None
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                cand = 1 + dfs(nr, nc, depth + 1)
                if cand > best:
                    best, via = cand, (nr, nc)
        memo[r][c] = best
        note = f"tốt nhất qua {via}" if via else "không có ô lớn hơn"
        print(f"{'    ' * depth}xong ({r},{c})={matrix[r][c]}: memo = {best}  ({note})")
        return best

    answer = 0
    for r in range(m):
        for c in range(n):
            answer = max(answer, dfs(r, c, 0))
    print_grid(memo, "Bảng memo cuối cùng:")
    print("Đáp án =", answer)
    return answer


trace_lip_memo(ex1)

[vòng ngoài] vào (0,0)=9
xong (0,0)=9: memo = 1  (không có ô lớn hơn)
[vòng ngoài] vào (0,1)=9
xong (0,1)=9: memo = 1  (không có ô lớn hơn)
[vòng ngoài] vào (0,2)=4
    vào (1,2)=8
    xong (1,2)=8: memo = 1  (không có ô lớn hơn)
    (0,1)=9: đã có memo = 1 -> dùng lại
xong (0,2)=4: memo = 2  (tốt nhất qua (1, 2))
[vòng ngoài] vào (1,0)=6
    (0,0)=9: đã có memo = 1 -> dùng lại
xong (1,0)=6: memo = 2  (tốt nhất qua (0, 0))
[vòng ngoài] vào (1,1)=6
    (0,1)=9: đã có memo = 1 -> dùng lại
    (1,2)=8: đã có memo = 1 -> dùng lại
xong (1,1)=6: memo = 2  (tốt nhất qua (0, 1))
[vòng ngoài] (1,2)=8: đã có memo = 1 -> dùng lại
[vòng ngoài] vào (2,0)=2
    (1,0)=6: đã có memo = 2 -> dùng lại
xong (2,0)=2: memo = 3  (tốt nhất qua (1, 0))
[vòng ngoài] vào (2,1)=1
    (1,1)=6: đã có memo = 2 -> dùng lại
    (2,0)=2: đã có memo = 3 -> dùng lại
xong (2,1)=1: memo = 4  (tốt nhất qua (2, 0))
[vòng ngoài] vào (2,2)=1
    (1,2)=8: đã có memo = 1 -> dùng lại
xong (2,2)=1: memo = 2  (tốt nhất qua (1, 2))


4

### 6.1 Đọc bảng memo và **truy vết** lại chính đường đi

Bảng memo cuối cùng của Example 1:

```text
  1  1  2
  2  2  1
  3  4  2
```

Mỗi số là "từ ô này, đi được tối đa bao nhiêu ô". Đáp án là số lớn nhất, `4`, tại ô `(2,1)`.

Muốn biết **đường đi cụ thể** (không chỉ độ dài), ta đi ngược lại công thức: đứng ở ô có `memo = k`, chắc chắn có một hàng xóm **lớn hơn** có `memo = k - 1` (chính hàng xóm đã tạo ra `max`). Bước sang đó, lặp lại cho tới khi `memo = 1`.

```text
(2,1)=1 [memo 4] -> (2,0)=2 [memo 3] -> (1,0)=6 [memo 2] -> (0,0)=9 [memo 1]
```

Ra đúng `[1, 2, 6, 9]` như đề. Với Example 2, ô có memo lớn nhất đầu tiên (theo thứ tự hàng, cột) là `(0,0)=3`, truy vết ra `[3, 4, 5, 6]` — cũng khớp đề.

In [14]:
def recover_path(matrix, memo):
    m, n = len(matrix), len(matrix[0])
    # 1) tìm ô có memo lớn nhất (gặp trước thì lấy trước)
    r, c = 0, 0
    for i in range(m):
        for j in range(n):
            if memo[i][j] > memo[r][c]:
                r, c = i, j
    # 2) đi theo hàng xóm lớn hơn có memo nhỏ hơn đúng 1
    path = [(r, c)]
    while memo[r][c] > 1:
        for nr, nc in neighbors(matrix, r, c):
            if matrix[nr][nc] > matrix[r][c] and memo[nr][nc] == memo[r][c] - 1:
                r, c = nr, nc
                break
        path.append((r, c))
    return path


for name, mat in [("Example 1", ex1), ("Example 2", ex2)]:
    ans, memo_grid = lip_memo(mat, return_memo=True)
    p = recover_path(mat, memo_grid)
    assert is_increasing_path(mat, p) and len(p) == ans
    print(f"{name}: ô {p} -> giá trị {[mat[r][c] for r, c in p]}")

Example 1: ô [(2, 1), (2, 0), (1, 0), (0, 0)] -> giá trị [1, 2, 6, 9]
Example 2: ô [(0, 0), (0, 1), (0, 2), (1, 2)] -> giá trị [3, 4, 5, 6]


## 7. Code nộp LeetCode (Cách 2)

Chuyển `lip_memo` sang đúng khuôn của LeetCode:

- Tên class `Solution`, method `longestIncreasingPath(self, matrix: List[List[int]]) -> int`.
- Hàm `dfs` định nghĩa **bên trong** method: nó "nhìn thấy" `matrix`, `memo`, `m`, `n` của lần gọi hiện tại, và mỗi test case có bảng memo mới (không dính dữ liệu cũ).
- Lưu `v = matrix[r][c]` vào biến cục bộ cho gọn và nhanh hơn một chút.

**Về độ sâu đệ quy.** Python mặc định chỉ cho khoảng `1000` tầng đệ quy (`sys.getrecursionlimit()`). Đường tăng có thể dài tới `m · n = 40 000` ô, và DFS có thể phải đi sâu gần chừng đó tầng. Vì vậy ta gọi `sys.setrecursionlimit(...)` để nới giới hạn. Mức an toàn của việc đệ quy rất sâu còn tùy phiên bản Python và môi trường chạy; nếu không muốn phụ thuộc vào điều đó, hãy nộp bản **không đệ quy** ở mục 8.5 (Kahn) hoặc Cách 4.

Để kiểm tra trường hợp "đường cực dài", ta tạo lưới `deep_snake(m, n)`: các hàng chẵn là một con rắn có giá trị `1, 2, 3, ...` chạy zig-zag, các hàng lẻ là "tường" giá trị `0` chỉ chừa một ô nối ở chỗ rắn quay đầu. Mỗi ô trên thân rắn chỉ có **đúng một** hàng xóm lớn hơn (ô kế tiếp), nên DFS từ đầu rắn buộc phải đi sâu hết chiều dài con rắn. Với `200 × 200`, đường dài nhất là `20 100` ô (gồm cả ô tường `0` nằm ngay dưới đầu rắn).

In [15]:
class Solution:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 100000))   # đường tăng có thể rất dài
        m, n = len(matrix), len(matrix[0])
        memo = [[0] * n for _ in range(m)]
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))

        def dfs(r: int, c: int) -> int:
            if memo[r][c]:
                return memo[r][c]
            v = matrix[r][c]
            best = 1
            for dr, dc in dirs:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > v:
                    best = max(best, 1 + dfs(nr, nc))
            memo[r][c] = best
            return best

        return max(dfs(r, c) for r in range(m) for c in range(n))


def deep_snake(m, n):
    mat = [[0] * n for _ in range(m)]
    v = 1
    for r in range(0, m, 2):
        cols = range(n) if (r // 2) % 2 == 0 else range(n - 1, -1, -1)
        for c in cols:
            mat[r][c] = v
            v += 1
        if r + 2 < m:                                   # ô nối xuống hàng rắn tiếp theo
            mat[r + 1][n - 1 if (r // 2) % 2 == 0 else 0] = v
            v += 1
    return mat


sol = Solution()
# Ví dụ của đề
assert sol.longestIncreasingPath(ex1) == 4
assert sol.longestIncreasingPath(ex2) == 4
assert sol.longestIncreasingPath(ex3) == 1
# Edge case
assert sol.longestIncreasingPath([[7, 7], [7, 7]]) == 1            # toàn bằng nhau
assert sol.longestIncreasingPath([[1, 2, 3, 4, 5]]) == 5           # một hàng
assert sol.longestIncreasingPath([[5], [4], [3], [2], [1]]) == 5   # một cột
assert sol.longestIncreasingPath([[0, 2**31 - 1]]) == 2            # giá trị cực trị
assert sol.longestIncreasingPath([[1, 5, 2]]) == 2                 # 1 và 2 không kề nhau
print_grid(deep_snake(5, 4), "deep_snake(5, 4):")
assert sol.longestIncreasingPath(deep_snake(5, 4)) == 15
t0 = time.perf_counter()
print("deep_snake(200, 200):", sol.longestIncreasingPath(deep_snake(200, 200)),
      f"(≈ {(time.perf_counter() - t0) * 1000:.0f} ms)")
print("Solution PASS")

deep_snake(5, 4):
  1   2   3   4
  0   0   0   5
  9   8   7   6
 10   0   0   0
 11  12  13  14
deep_snake(200, 200): 20100 (≈ 33 ms)
Solution PASS


## 8. Cách 3 — Topological sort (Kahn): bóc từng lớp, không đệ quy

### 8.1 Nhìn lưới như một **đồ thị có hướng**

- Mỗi ô là một **đỉnh** (node).
- Vẽ **mũi tên** (cạnh có hướng, directed edge) từ ô `A` sang hàng xóm `B` khi `matrix[B] > matrix[A]`. Đây chính là bảng mũi tên ở mục 2.2.
- **Bậc ra** (out-degree) của một ô = số mũi tên đi ra = số hàng xóm **lớn hơn** nó.
- **Bậc vào** (in-degree) = số mũi tên đi vào = số hàng xóm **nhỏ hơn** nó.
- Một đường tăng = một dãy ô đi theo mũi tên.

Đồ thị này **không có chu trình** (cycle — đi theo mũi tên mà quay về chỗ cũ): đi theo mũi tên thì giá trị tăng ngặt, quay về chỗ cũ nghĩa là `v < ... < v`, vô lý. Đồ thị có hướng không chu trình gọi là **DAG** (Directed Acyclic Graph). Bài 329 chính là: **tìm đường dài nhất trong một DAG** — bài có lời giải tuyến tính.

Ô có **bậc ra 0** là ô không còn hàng xóm nào lớn hơn — gọi vui là các **"đỉnh núi"**: mọi đường tăng đều phải kết thúc ở một đỉnh núi.

In [16]:
def degree_grids(matrix):
    m, n = len(matrix), len(matrix[0])
    out_deg = [[0] * n for _ in range(m)]
    in_deg = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            for nr, nc in neighbors(matrix, r, c):
                if matrix[nr][nc] > matrix[r][c]:     # mũi tên (r,c) -> (nr,nc)
                    out_deg[r][c] += 1
                    in_deg[nr][nc] += 1
    return out_deg, in_deg


out_deg, in_deg = degree_grids(ex1)
print_grid(ex1, "Example 1:")
print_grid(out_deg, "bậc ra  (số hàng xóm LỚN hơn):")
print_grid(in_deg, "bậc vào (số hàng xóm NHỎ hơn):")
print("tổng bậc ra =", sum(map(sum, out_deg)), "= tổng bậc vào =", sum(map(sum, in_deg)), "= số mũi tên")

Example 1:
  9   9   4
  6   6   8
  2   1   1
bậc ra  (số hàng xóm LỚN hơn):
  0   0   2
  1   2   0
  1   2   1
bậc vào (số hàng xóm NHỎ hơn):
  1   2   0
  1   1   3
  1   0   0
tổng bậc ra = 9 = tổng bậc vào = 9 = số mũi tên


### 8.2 Ý tưởng "bóc vỏ hành" trên một DAG nhỏ, và hàng đợi `deque`

Xét DAG nhỏ (mũi tên `X → Y`):

```text
A → B → D → F
A → C → D
C → E
```

Chuỗi mũi tên dài nhất là `A → B → D → F` (hoặc `A → C → D → F`), gồm **4 đỉnh**.

Thuật toán **Kahn** (một cách làm *topological sort* — sắp xếp các đỉnh sao cho mũi tên luôn đi "xuôi") ở dạng **bóc từng lớp**:

1. **Lớp 1** = mọi đỉnh có bậc ra 0 (không còn mũi tên đi ra): `E`, `F`.
2. "Bóc" lớp đó khỏi đồ thị: với mỗi đỉnh vừa bóc, mọi đỉnh **chỉ vào nó** bị giảm bậc ra đi 1. Đỉnh nào về 0 thì thuộc **lớp kế tiếp**.
3. Lặp tới khi hết đỉnh. **Số lớp = số đỉnh trên chuỗi dài nhất.**

Vì sao? Một đỉnh chỉ được bóc khi mọi đỉnh nó chỉ tới đã bị bóc, tức là nó nằm ở lớp ngay sau lớp muộn nhất trong số các đỉnh nó chỉ tới. Nên `lớp(X) = 1 + max(lớp của các đỉnh X chỉ tới)` — y hệt công thức `longest_from` của mục 3.1!

(Sách giáo khoa thường trình bày Kahn với **bậc vào** — bắt đầu từ đỉnh không có mũi tên đi vào. Hai chiều là đối xứng; Bài tập 4 sẽ cho bạn làm chiều đó.)

Công cụ: **hàng đợi** (queue) — vào trước ra trước (FIFO). Python dùng `collections.deque`:

- `q.append(x)`: thêm vào **cuối** hàng; `q.popleft()`: lấy ra ở **đầu** hàng, nhanh `O(1)` (còn `list.pop(0)` chậm `O(n)`).
- Mẹo xử lý **theo lớp**: chụp `len(q)` ở đầu mỗi vòng `while` — đó đúng bằng số phần tử của lớp hiện tại; phần tử mới thêm vào trong lúc bóc sẽ thuộc lớp sau.

In [17]:
q = deque()
q.append("A"); q.append("B"); q.append("C")
print("hàng đợi:", list(q), "| popleft ->", q.popleft(), "| còn:", list(q))

graph = {"A": ["B", "C"], "B": ["D"], "C": ["D", "E"], "D": ["F"], "E": [], "F": []}
parents = {v: [] for v in graph}                 # ai chỉ vào tôi?
for u in graph:
    for v in graph[u]:
        parents[v].append(u)
out_degree = {v: len(graph[v]) for v in graph}
print("bậc ra:", out_degree)

q = deque(v for v in graph if out_degree[v] == 0)
layer = 0
while q:
    layer += 1
    size = len(q)                                # số đỉnh của lớp hiện tại
    current = []
    for _ in range(size):
        v = q.popleft()
        current.append(v)
        for u in parents[v]:                     # bóc v -> giảm bậc ra của ai chỉ vào v
            out_degree[u] -= 1
            if out_degree[u] == 0:
                q.append(u)
    print(f"lớp {layer}: {current}")
print("số lớp =", layer, "= số đỉnh trên chuỗi dài nhất A -> B -> D -> F")

hàng đợi: ['A', 'B', 'C'] | popleft -> A | còn: ['B', 'C']
bậc ra: {'A': 2, 'B': 1, 'C': 2, 'D': 1, 'E': 0, 'F': 0}
lớp 1: ['E', 'F']
lớp 2: ['D']
lớp 3: ['B', 'C']
lớp 4: ['A']
số lớp = 4 = số đỉnh trên chuỗi dài nhất A -> B -> D -> F


### 8.3 Áp vào lưới

Trên lưới, "ai chỉ vào ô `(r, c)`" chính là các **hàng xóm nhỏ hơn** nó — không cần dựng `parents` riêng, chỉ cần nhìn 4 hướng.

```text
1. Tính out_deg[r][c] = số hàng xóm LỚN hơn.
2. Đưa mọi ô có out_deg = 0 (đỉnh núi) vào hàng đợi.     -> lớp 1
3. Lặp khi hàng đợi còn phần tử:
       layers += 1
       bóc hết các ô của lớp hiện tại; với mỗi ô (r, c):
           với mỗi hàng xóm NHỎ hơn (nr, nc):           # (nr,nc) -> (r,c) là một mũi tên
               out_deg[nr][nc] -= 1
               nếu về 0: đưa (nr, nc) vào hàng đợi      # thuộc lớp sau
4. Trả về layers.
```

Lớp của một ô đúng bằng `memo` của nó ở Cách 2 (đường tăng dài nhất **bắt đầu** từ ô đó). Cell dưới kiểm chứng bằng cách ghi lại lớp của từng ô rồi so với bảng memo.

Ưu điểm lớn: **không có đệ quy** nên không lo giới hạn độ sâu; mỗi ô vào hàng đợi đúng một lần.

In [18]:
def lip_topo(matrix, return_layers=False):
    m, n = len(matrix), len(matrix[0])
    out_deg = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                    out_deg[r][c] += 1

    q = deque()
    for r in range(m):
        for c in range(n):
            if out_deg[r][c] == 0:                 # đỉnh núi -> lớp 1
                q.append((r, c))

    layer_of = [[0] * n for _ in range(m)]
    layers = 0
    while q:
        layers += 1
        for _ in range(len(q)):                    # bóc đúng các ô của lớp hiện tại
            r, c = q.popleft()
            layer_of[r][c] = layers
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] < matrix[r][c]:
                    out_deg[nr][nc] -= 1
                    if out_deg[nr][nc] == 0:
                        q.append((nr, nc))
    return (layers, layer_of) if return_layers else layers


for name, mat in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3)]:
    layers, layer_of = lip_topo(mat, return_layers=True)
    _, memo_grid = lip_memo(mat, return_memo=True)
    print(f"{name}: số lớp = {layers}; lớp của từng ô == bảng memo? {layer_of == memo_grid}")

Example 1: số lớp = 4; lớp của từng ô == bảng memo? True
Example 2: số lớp = 4; lớp của từng ô == bảng memo? True
Example 3: số lớp = 1; lớp của từng ô == bảng memo? True


### 8.4 Trace Kahn trên Example 1

Bậc ra ban đầu (in ở mục 8.1):

```text
  0  0  2
  1  2  0
  1  2  1
```

| Lớp | Các ô bị bóc | Giảm bậc ra của hàng xóm nhỏ hơn | Ô về 0 → lớp sau |
|:---:|---|---|---|
| 1 | (0,0)=9, (0,1)=9, (1,2)=8 | (1,0): 1→0; (1,1): 2→1; (0,2): 2→1; (0,2): 1→0; (2,2): 1→0; (1,1): 1→0 | (1,0), (0,2), (2,2), (1,1) |
| 2 | (1,0)=6, (0,2)=4, (2,2)=1, (1,1)=6 | (2,0): 1→0; (2,1): 2→1 | (2,0) |
| 3 | (2,0)=2 | (2,1): 1→0 | (2,1) |
| 4 | (2,1)=1 | (không có hàng xóm nhỏ hơn) | — |

Hết hàng đợi sau **4 lớp** → đáp án `4`. Ô `(2,1)=1` ở lớp 4: nó là "chân núi" xa đỉnh nhất.

Để ý dòng lớp 1: ô `(0,2)=4` bị giảm **hai lần** (một lần khi bóc `(0,1)=9`, một lần khi bóc `(1,2)=8`) — nó chỉ vào lớp sau khi **cả hai** ô lớn hơn nó đều đã bị bóc.

In [19]:
def trace_lip_topo(matrix):
    m, n = len(matrix), len(matrix[0])
    out_deg, _ = degree_grids(matrix)
    q = deque((r, c) for r in range(m) for c in range(n) if out_deg[r][c] == 0)
    layers = 0
    while q:
        layers += 1
        size = len(q)
        removed = []
        changes = []
        for _ in range(size):
            r, c = q.popleft()
            removed.append(f"({r},{c})={matrix[r][c]}")
            for nr, nc in neighbors(matrix, r, c):
                if matrix[nr][nc] < matrix[r][c]:
                    out_deg[nr][nc] -= 1
                    changes.append(f"({nr},{nc}):{out_deg[nr][nc] + 1}->{out_deg[nr][nc]}")
                    if out_deg[nr][nc] == 0:
                        q.append((nr, nc))
        print(f"Lớp {layers}: bóc {', '.join(removed)}")
        print(f"   giảm bậc ra: {'; '.join(changes) if changes else '(không có)'}")
        print(f"   vào lớp sau: {[f'({r},{c})' for r, c in q] if q else '(hết)'}")
    print("Số lớp =", layers)
    return layers


trace_lip_topo(ex1)

Lớp 1: bóc (0,0)=9, (0,1)=9, (1,2)=8
   giảm bậc ra: (1,0):1->0; (1,1):2->1; (0,2):2->1; (0,2):1->0; (2,2):1->0; (1,1):1->0
   vào lớp sau: ['(1,0)', '(0,2)', '(2,2)', '(1,1)']
Lớp 2: bóc (1,0)=6, (0,2)=4, (2,2)=1, (1,1)=6
   giảm bậc ra: (2,0):1->0; (2,1):2->1
   vào lớp sau: ['(2,0)']
Lớp 3: bóc (2,0)=2
   giảm bậc ra: (2,1):1->0
   vào lớp sau: ['(2,1)']
Lớp 4: bóc (2,1)=1
   giảm bậc ra: (không có)
   vào lớp sau: (hết)
Số lớp = 4


4

### 8.5 Bản LeetCode của Cách 3 (không đệ quy)

Đây là bản nên chọn nếu bạn muốn **chắc chắn** không gặp vấn đề độ sâu đệ quy. Logic y hệt `lip_topo`, chỉ bỏ phần ghi `layer_of`.

In [20]:
class SolutionTopo:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:
        m, n = len(matrix), len(matrix[0])
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        out_deg = [[0] * n for _ in range(m)]
        for r in range(m):
            for c in range(n):
                v = matrix[r][c]
                for dr, dc in dirs:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > v:
                        out_deg[r][c] += 1

        q = deque((r, c) for r in range(m) for c in range(n) if out_deg[r][c] == 0)
        layers = 0
        while q:
            layers += 1
            for _ in range(len(q)):
                r, c = q.popleft()
                v = matrix[r][c]
                for dr, dc in dirs:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] < v:
                        out_deg[nr][nc] -= 1
                        if out_deg[nr][nc] == 0:
                            q.append((nr, nc))
        return layers


st = SolutionTopo()
assert st.longestIncreasingPath(ex1) == 4
assert st.longestIncreasingPath(ex2) == 4
assert st.longestIncreasingPath(ex3) == 1
assert st.longestIncreasingPath([[7, 7], [7, 7]]) == 1
assert st.longestIncreasingPath([[1, 2, 3, 4, 5]]) == 5
assert st.longestIncreasingPath([[1, 5, 2]]) == 2
assert st.longestIncreasingPath(deep_snake(200, 200)) == 20100
print("SolutionTopo PASS")

SolutionTopo PASS


## 9. Cách 4 — Sắp xếp các ô theo giá trị rồi DP

### 9.1 Ý tưởng "ô nhỏ trước, ô lớn sau"

Đổi góc nhìn: thay vì "đường dài nhất **bắt đầu** tại ô", xét

> `dp[r][c]` = độ dài đường tăng dài nhất **kết thúc** tại ô `(r, c)`.

Đường tăng kết thúc tại `(r, c)` hoặc chỉ có mình nó, hoặc đến từ một hàng xóm **nhỏ hơn**:

```text
dp[r][c] = 1 + max( dp[hàng xóm nhỏ hơn] )      (= 1 nếu không có hàng xóm nhỏ hơn)
```

Muốn tính `dp[r][c]` thì mọi hàng xóm nhỏ hơn phải tính xong trước. Cách đơn giản nhất để bảo đảm điều đó: **xử lý các ô theo giá trị tăng dần**. Không cần đệ quy, không cần hàng đợi — chỉ cần `sorted`.

Còn các ô **bằng nhau**? Giữa chúng không có mũi tên (bằng nhau không được đi), nên xử lý ô nào trước cũng được.

Công cụ: `sorted` một list các **tuple** `(giá trị, r, c)`. Python so sánh tuple **từ trái sang phải**: so phần tử đầu (giá trị) trước, bằng nhau mới so tới `r`, rồi `c`. Nên sắp tuple là sắp theo giá trị tăng dần.

In [21]:
cells = [(ex1[r][c], r, c) for r in range(3) for c in range(3)]
print("trước khi sắp:", cells)
print("sau khi sắp  :", sorted(cells))
print("so sánh tuple:", (1, 2, 1) < (1, 2, 2), (2, 0, 0) < (1, 9, 9))

trước khi sắp: [(9, 0, 0), (9, 0, 1), (4, 0, 2), (6, 1, 0), (6, 1, 1), (8, 1, 2), (2, 2, 0), (1, 2, 1), (1, 2, 2)]
sau khi sắp  : [(1, 2, 1), (1, 2, 2), (2, 2, 0), (4, 0, 2), (6, 1, 0), (6, 1, 1), (8, 1, 2), (9, 0, 0), (9, 0, 1)]
so sánh tuple: True False


### 9.2 Code và trace trên Example 1

Hàm `lip_sort_dp` làm đúng 3 bước: tạo list `(giá trị, r, c)` → `sorted` → duyệt và cập nhật `dp` từ các hàng xóm nhỏ hơn. Hàm `trace_lip_sort_dp` in từng ô theo thứ tự xử lý.

Dự đoán bằng tay (thứ tự sau khi sắp: `1(2,1), 1(2,2), 2(2,0), 4(0,2), 6(1,0), 6(1,1), 8(1,2), 9(0,0), 9(0,1)`):

| Ô | Giá trị | Hàng xóm nhỏ hơn (dp đã tính) | dp |
|---|:---:|---|:---:|
| (2,1) | 1 | không có | 1 |
| (2,2) | 1 | không có (`(2,1)` bằng nhau, không tính) | 1 |
| (2,0) | 2 | (2,1)=1 → 1 | 2 |
| (0,2) | 4 | không có | 1 |
| (1,0) | 6 | (2,0)=2 → 2 | 3 |
| (1,1) | 6 | (2,1)=1 → 1 | 2 |
| (1,2) | 8 | (0,2)=4 → 1, (2,2)=1 → 1, (1,1)=6 → 2 | 3 |
| (0,0) | 9 | (1,0)=6 → 3 | **4** |
| (0,1) | 9 | (1,1)=6 → 2, (0,2)=4 → 1 | 3 |

Đáp án `max(dp) = 4`, tại ô `(0,0)=9`: đường `1 → 2 → 6 → 9` **kết thúc** ở đó. Bảng `dp` khác bảng memo của Cách 2 (vì một bên "bắt đầu tại", một bên "kết thúc tại"), nhưng **số lớn nhất** luôn như nhau: cùng một đường dài nhất, chỉ đếm từ hai đầu.

In [22]:
def lip_sort_dp(matrix):
    m, n = len(matrix), len(matrix[0])
    cells = sorted((matrix[r][c], r, c) for r in range(m) for c in range(n))
    dp = [[1] * n for _ in range(m)]          # dp[r][c] = đường tăng dài nhất KẾT THÚC tại (r, c)
    for v, r, c in cells:
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] < v:
                dp[r][c] = max(dp[r][c], dp[nr][nc] + 1)
    return max(max(row) for row in dp)


def trace_lip_sort_dp(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[1] * n for _ in range(m)]
    for v, r, c in sorted((matrix[r][c], r, c) for r in range(m) for c in range(n)):
        smaller = []
        for nr, nc in neighbors(matrix, r, c):
            if matrix[nr][nc] < v:
                smaller.append(f"({nr},{nc})={matrix[nr][nc]} dp {dp[nr][nc]}")
                dp[r][c] = max(dp[r][c], dp[nr][nc] + 1)
        print(f"v={v} ô ({r},{c}): nhỏ hơn: {', '.join(smaller) if smaller else 'không có'} -> dp = {dp[r][c]}")
    print_grid(dp, "Bảng dp (kết thúc tại mỗi ô):")
    return max(max(row) for row in dp)


print("Đáp án =", trace_lip_sort_dp(ex1))
print("Example 2:", lip_sort_dp(ex2), "| Example 3:", lip_sort_dp(ex3))

v=1 ô (2,1): nhỏ hơn: không có -> dp = 1
v=1 ô (2,2): nhỏ hơn: không có -> dp = 1
v=2 ô (2,0): nhỏ hơn: (2,1)=1 dp 1 -> dp = 2
v=4 ô (0,2): nhỏ hơn: không có -> dp = 1
v=6 ô (1,0): nhỏ hơn: (2,0)=2 dp 2 -> dp = 3
v=6 ô (1,1): nhỏ hơn: (2,1)=1 dp 1 -> dp = 2
v=8 ô (1,2): nhỏ hơn: (0,2)=4 dp 1, (2,2)=1 dp 1, (1,1)=6 dp 2 -> dp = 3
v=9 ô (0,0): nhỏ hơn: (1,0)=6 dp 3 -> dp = 4
v=9 ô (0,1): nhỏ hơn: (1,1)=6 dp 2, (0,2)=4 dp 1 -> dp = 3
Bảng dp (kết thúc tại mỗi ô):
  4   3   1
  3   2   3
  2   1   1
Đáp án = 4
Example 2: 4 | Example 3: 1


### 9.3 Nhận xét về Cách 4

- **Không đệ quy, không hàng đợi**, code rất ngắn — dễ viết lại trong phòng thi.
- Chi phí chính là `sorted`: `O(N log N)` với `N = m · n`, chậm hơn `O(N)` của Cách 2/3 một chút về lý thuyết, nhưng thực tế vẫn rất nhanh với `N = 40 000`.
- Nếu sắp theo giá trị **giảm dần** và dùng hàng xóm **lớn hơn**, ta tính được đúng bảng "bắt đầu tại" — trùng khớp bảng memo của Cách 2. Thực chất Cách 2 chỉ là để **đệ quy tự tìm** một thứ tự hợp lệ (ô lớn xong trước ô nhỏ), còn Cách 4 **chủ động sắp xếp** ra thứ tự đó. Cách 3 thì tìm thứ tự bằng cách bóc lớp. Ba cách, một bản chất: **DP trên DAG theo một thứ tự topo**.
- Ý tưởng "sắp các phần tử rồi DP theo thứ tự" tái xuất hiện ở nhiều bài: *Longest String Chain*, *Maximum Strictly Increasing Cells in a Matrix*...

## 10. Đối chiếu chéo các cách trên lưới ngẫu nhiên

Brute force chậm nhưng **chắc chắn đúng** (thử mọi đường), nên dùng nó làm "trọng tài" trên các lưới nhỏ. Lưới ngẫu nhiên kích thước `1..4 × 1..4`, giá trị chỉ trong `0..4` để có **thật nhiều ô bằng nhau** — chỗ dễ sai nhất. `random.seed(329)` cố định để lần chạy nào cũng ra cùng bộ test.

Sau đó đo thời gian 3 cách nhanh trên lưới `200 × 200` (đúng giới hạn đề): một lưới giá trị ngẫu nhiên lớn, một lưới giá trị nhỏ `0..9` (rất nhiều ô bằng nhau), và `deep_snake` (đường cực dài). Con số cụ thể tùy máy; chỉ cần nhìn **độ lớn**.

In [23]:
random.seed(329)
sol, st = Solution(), SolutionTopo()
for _ in range(400):
    m, n = random.randint(1, 4), random.randint(1, 4)
    mat = [[random.randint(0, 4) for _ in range(n)] for _ in range(m)]
    expected = lip_bruteforce(mat)
    got = [lip_memo(mat), lip_memo_cache(mat), lip_topo(mat), lip_sort_dp(mat),
           sol.longestIncreasingPath(mat), st.longestIncreasingPath(mat)]
    assert all(g == expected for g in got), (mat, expected, got)
print("400 lưới ngẫu nhiên: 6 cài đặt đều khớp brute force")

sys.setrecursionlimit(max(sys.getrecursionlimit(), 100000))
random.seed(2026)
tests = {
    "ngẫu nhiên 0..2^31-1": [[random.randint(0, 2**31 - 1) for _ in range(200)] for _ in range(200)],
    "ngẫu nhiên 0..9     ": [[random.randint(0, 9) for _ in range(200)] for _ in range(200)],
    "deep_snake          ": deep_snake(200, 200),
}
for label, mat in tests.items():
    line = []
    for name, fn in [("memo", lip_memo), ("topo", lip_topo), ("sort", lip_sort_dp)]:
        t0 = time.perf_counter()
        ans = fn(mat)
        line.append(f"{name}={ans} ({(time.perf_counter() - t0) * 1000:4.0f} ms)")
    print(label, "|", " | ".join(line))

400 lưới ngẫu nhiên: 6 cài đặt đều khớp brute force


ngẫu nhiên 0..2^31-1 | memo=12 (  41 ms) | topo=12 (  63 ms) | sort=12 (  54 ms)


ngẫu nhiên 0..9      | memo=8 (  33 ms) | topo=8 (  61 ms) | sort=8 (  43 ms)


deep_snake           | memo=20100 (  40 ms) | topo=20100 (  60 ms) | sort=20100 (  57 ms)


## 11. Vì sao các cách đúng?

Gọi `f(r, c)` = độ dài đường tăng dài nhất **bắt đầu** tại `(r, c)`.

**Cách 2 (DFS + memo).**

- *Công thức đúng*: mọi đường tăng bắt đầu tại `(r, c)` hoặc chỉ gồm `(r, c)`, hoặc là `(r, c)` nối với một đường tăng bắt đầu tại một hàng xóm lớn hơn. Ngược lại, ghép `(r, c)` vào trước bất kỳ đường tăng nào bắt đầu tại hàng xóm lớn hơn đều được một đường tăng hợp lệ (không lặp ô — mục 2.5). Nên `f(r, c) = 1 + max f(hàng xóm lớn hơn)`, hoặc `1`.
- *Đệ quy luôn dừng*: mỗi lời gọi đi tới ô có giá trị **lớn hơn hẳn**; một dãy giá trị tăng ngặt trên hữu hạn ô không thể kéo dài mãi. Chứng minh `dfs` trả đúng `f` bằng **quy nạp theo giá trị từ lớn xuống nhỏ**: các ô lớn nhất không có hàng xóm lớn hơn, trả `1` — đúng; một ô bất kỳ chỉ cần kết quả của các ô lớn hơn nó, mà những ô đó đã đúng theo giả thiết quy nạp.
- *Memo hợp lệ*: `f(r, c)` chỉ phụ thuộc `(r, c)` (không có `visited`), nên lưu lại và dùng lại không làm thay đổi kết quả.
- *Đáp án*: đường dài nhất bắt đầu ở một ô nào đó, nên đáp án `= max f(r, c)` trên mọi ô.

**Cách 3 (Kahn).** Invariant: *lớp thứ `k` gồm đúng những ô có `f = k`*. Một ô được đưa vào hàng đợi đúng lúc ô lớn hơn **cuối cùng** của nó bị bóc, nên nó nằm ở lớp ngay sau lớp lớn nhất của các ô lớn hơn: `lớp(ô) = 1 + max lớp(ô lớn hơn)` — cùng công thức, cùng điểm xuất phát (đỉnh núi có lớp 1 = `f` là 1). Vì là DAG, mọi ô cuối cùng đều được bóc. Số lớp = `max f`.

**Cách 4 (sort + DP).** Invariant: *khi xử lý một ô giá trị `v`, mọi ô có giá trị `< v` đã có `dp` cuối cùng*. Do đó `dp[r][c] = 1 + max dp(hàng xóm nhỏ hơn)` được tính bằng các giá trị đã đúng. `dp` đo "kết thúc tại", `f` đo "bắt đầu tại", nhưng cùng một tập đường đi nên `max dp = max f`.

## 12. Độ phức tạp và so sánh các cách

`N = m · n` = số ô (tối đa 40 000). Mỗi ô có tối đa 4 hàng xóm.

| Cách | Ý tưởng | Time | Space | Đệ quy? | Ghi chú |
|---|---|---|---|:---:|---|
| 1. DFS thuần | thử mọi đường tăng từ mọi ô | **hàm mũ** (lưới `i+j`: cỡ $\binom{m+n-2}{m-1}$ đường) | `O(N)` call stack | có | chỉ dùng làm trọng tài test |
| 2. **DFS + memo** | mỗi ô tính 1 lần, nhớ lại | **`O(N)`** | `O(N)` memo + `O(N)` call stack (tệ nhất) | có | **cách chính**, ngắn, tự nhiên |
| 3. Kahn (bóc lớp) | đỉnh núi trước, số lớp = đáp án | **`O(N)`** | `O(N)` bậc ra + hàng đợi | không | an toàn với đường cực dài |
| 4. Sort + DP | ô nhỏ trước, `dp` kết thúc tại | `O(N log N)` | `O(N)` | không | dễ viết nhất khi đã quen |

Vì sao Cách 2 là `O(N)`: mỗi ô được "tính thật" đúng một lần, mỗi lần xét 4 hướng → tối đa `4N` lần kiểm tra; mỗi lời gọi "tra sổ" tốn `O(1)` và có không quá `N + 4N` lời gọi như vậy. Bộ nhớ: bảng memo `N` ô cộng call stack có thể sâu tới cỡ `N` tầng khi đường tăng rất dài (ví dụ `deep_snake`).

Vì sao Cách 3 là `O(N)`: tính bậc ra mất `4N`; mỗi ô vào và ra hàng đợi đúng một lần, mỗi lần xét 4 hướng.

## 13. Lỗi thường gặp

### Lỗi 1: Dùng `>=` thay vì `>` (cho phép đi sang ô bằng nhau)

```python
if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] >= matrix[r][c]:   # SAI
```

Đề yêu cầu **tăng ngặt**. Với `>=`, hai ô kề nhau bằng nhau (như `(0,0)=9` và `(0,1)=9`) gọi qua gọi lại nhau mãi: `dfs(0,0) → dfs(0,1) → dfs(0,0) → ...`. Memo **không cứu được**, vì `memo` chỉ được ghi khi một lời gọi **xong**, mà lời gọi nào cũng đang chờ lời gọi sau → `RecursionError`. (Kể cả nếu "vá" bằng `visited` cho khỏi treo thì đáp án vẫn sai: `1 → 2 → 6 → 6 → 8` dài 5 thay vì 4, và memo lại mất tác dụng.)

Cell dưới tạm hạ giới hạn đệ quy về 1000 để lỗi hiện ra nhanh, rồi khôi phục.

In [24]:
def lip_wrong_ge(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]

    def dfs(r, c):
        if memo[r][c]:
            return memo[r][c]
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] >= matrix[r][c]:   # SAI: >=
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n))


old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    lip_wrong_ge(ex1)
except RecursionError as e:
    print("lip_wrong_ge(ex1) -> RecursionError:", e)
finally:
    sys.setrecursionlimit(old_limit)
print("lip_memo(ex1)     ->", lip_memo(ex1))

lip_wrong_ge(ex1) -> RecursionError: maximum recursion depth exceeded
lip_memo(ex1)     -> 4


### Lỗi 2: Chỉ kiểm tra biên trên (`nr < m`), quên biên dưới (`0 <= nr`)

```python
if nr < m and nc < n and matrix[nr][nc] > matrix[r][c]:    # SAI: thiếu 0 <= nr, 0 <= nc
```

Ở Python, `nc = -1` **không** gây lỗi mà đọc phần tử **cuối hàng** (mục 2.2). Kết quả là code âm thầm cho phép "quấn vòng" từ mép trái sang mép phải. Ví dụ `[[1, 5, 2]]`: đáp án đúng là `2` (`1 → 5` hoặc `2 → 5`; ô `1` và ô `2` **không** kề nhau). Bản sai đi `1 →(quấn sang trái) 2 → 5` và trả `3`. Không có lỗi nào được báo — chính vì thế lỗi này rất nguy hiểm.

Sửa: luôn viết đủ `0 <= nr < m and 0 <= nc < n`.

In [25]:
def lip_wrong_bounds(matrix):
    m, n = len(matrix), len(matrix[0])

    @lru_cache(maxsize=None)
    def dfs(r, c):
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if nr < m and nc < n and matrix[nr][nc] > matrix[r][c]:   # SAI: thiếu biên dưới
                best = max(best, 1 + dfs(nr, nc))
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n))


print("[[1, 5, 2]] bản sai :", lip_wrong_bounds([[1, 5, 2]]))   # 3
print("[[1, 5, 2]] bản đúng:", lip_memo([[1, 5, 2]]))           # 2

[[1, 5, 2]] bản sai : 3
[[1, 5, 2]] bản đúng: 2


### Lỗi 3: Tạo bảng memo bằng `[[0] * n] * m`

Như mục 4.2, mọi hàng của bảng là **cùng một list**. Ghi `memo[0][2] = 2` thì `memo[1][2]`, `memo[2][2]` cũng "có memo = 2" dù chưa hề được tính. Với Example 1, bản sai trả `2` thay vì `4`, và bảng memo in ra ba hàng giống hệt nhau.

### Lỗi 4: Đặt `@lru_cache` lên một hàm **toàn cục** đọc ma trận từ biến toàn cục

Nếu viết `dfs` ở ngoài (mức module) và cho nó đọc một biến toàn cục `grid`, thì cache nhớ theo `(r, c)` **không biết** `grid` đã đổi. Gọi cho ma trận thứ hai cùng kích thước, mọi `(r, c)` đều "trúng cache" từ ma trận cũ → trả kết quả của ma trận cũ. Trên LeetCode, nhiều test chạy trong cùng một tiến trình, nên lỗi này có thể làm sai ở test thứ hai trở đi.

Sửa: định nghĩa `dfs` **bên trong** method (mỗi lần gọi có cache mới), hoặc gọi `dfs.cache_clear()` trước mỗi test.

In [26]:
def lip_wrong_alias(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n] * m                     # SAI: các hàng dùng chung một list

    def dfs(r, c):
        if memo[r][c]:
            return memo[r][c]
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n)), memo


ans, bad_memo = lip_wrong_alias(ex1)
print("Lỗi 3 - bản sai:", ans, "| bảng memo:", bad_memo, "| bản đúng:", lip_memo(ex1))

grid = None


@lru_cache(maxsize=None)
def dfs_global(r, c):                        # SAI: đọc grid toàn cục, cache không biết grid đổi
    best = 1
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] > grid[r][c]:
            best = max(best, 1 + dfs_global(nr, nc))
    return best


def lip_global(matrix):
    global grid
    grid = matrix
    return max(dfs_global(r, c) for r in range(len(matrix)) for c in range(len(matrix[0])))


snake3 = [[1, 2, 3], [6, 5, 4], [7, 8, 9]]
print("Lỗi 4 - test 1 (ex1)   :", lip_global(ex1))
print("Lỗi 4 - test 2 (snake3):", lip_global(snake3), "<- sai, đúng phải là", lip_memo(snake3))
dfs_global.cache_clear()
print("Sau cache_clear        :", lip_global(snake3))

Lỗi 3 - bản sai: 2 | bảng memo: [[1, 1, 2], [1, 1, 2], [1, 1, 2]] | bản đúng: 4
Lỗi 4 - test 1 (ex1)   : 4
Lỗi 4 - test 2 (snake3): 4 <- sai, đúng phải là 9
Sau cache_clear        : 9


### Lỗi 5: Chỉ bắt đầu từ ô `(0, 0)`

```python
return dfs(0, 0)      # SAI
```

Đường dài nhất có thể bắt đầu ở **bất kỳ** ô nào. Example 1: `dfs(0,0)` = `1` (ô `9` là đỉnh núi), trong khi đáp án là `4`. Phải lấy `max` trên mọi ô.

### Lỗi 6: Đếm **bước** thay vì đếm **ô**

Nếu base case trả `0` ("chưa đi bước nào") thì hàm đếm số cạnh, ra `3` cho Example 1. Đề đếm **số ô**: một ô đơn lẻ có độ dài `1` (Example 3). Sửa: `best = 1`.

### Lỗi 7: Có bảng memo nhưng quên ghi `memo[r][c] = best`

Code vẫn **đúng**, nhưng không bao giờ "trúng sổ" → quay về brute force hàm mũ → Time Limit Exceeded. Hãy test với lưới `i + j` cỡ `12 × 12` trở lên: bản đúng chạy tức thì, bản quên ghi sổ chạy rất lâu.

### Lỗi 8: Đệ quy quá sâu — và cái bẫy riêng của `lru_cache`

Với đường tăng rất dài (như `deep_snake`), giới hạn mặc định 1000 tầng không đủ → `RecursionError`. Gọi `sys.setrecursionlimit` giải quyết được cho bản dùng **bảng memo**. Nhưng với bản `@lru_cache`, mỗi lời gọi đi xuyên qua lớp bọc viết bằng C; trên các bản Python mới (3.12 trở đi) phần đệ quy đi qua code C bị một giới hạn **riêng**, nên bản `lru_cache` vẫn có thể `RecursionError` dù đã nới `setrecursionlimit`. Cell dưới chạy thử trên chính kernel của notebook này.

Cách an toàn tuyệt đối: dùng bản **không đệ quy** (Cách 3 hoặc Cách 4).

In [27]:
print("Python", sys.version.split()[0])
_, memo_ex1 = lip_memo(ex1, return_memo=True)
print("Lỗi 5 - chỉ dfs(0,0):", memo_ex1[0][0], "| đúng:", lip_memo(ex1))
print("Lỗi 6 - đếm bước    :", lip_memo(ex1) - 1, "| đúng:", lip_memo(ex1))

sys.setrecursionlimit(max(sys.getrecursionlimit(), 100000))
big_snake = deep_snake(200, 200)
print("bảng memo trên deep_snake(200,200):", lip_memo(big_snake))
try:
    print("lru_cache trên deep_snake(200,200):", lip_memo_cache(big_snake))
except RecursionError as e:
    print("lru_cache trên deep_snake(200,200): RecursionError -", e)
print("Kahn (không đệ quy)               :", lip_topo(big_snake))

Python 3.13.18
Lỗi 5 - chỉ dfs(0,0): 1 | đúng: 4
Lỗi 6 - đếm bước    : 3 | đúng: 4


bảng memo trên deep_snake(200,200): 20100
lru_cache trên deep_snake(200,200): RecursionError - maximum recursion depth exceeded while calling a Python object
Kahn (không đệ quy)               : 20100


## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận ra bài "DP trên DAG ngầm trong lưới"

- Đề cho **lưới**, đi **4 hướng**, và mỗi bước phải thỏa một điều kiện **có thứ tự ngặt** (tăng ngặt, giảm ngặt, "cao hơn hẳn"...). → Đồ thị ngầm là **DAG**, không cần `visited`.
- Câu hỏi là **dài nhất / đếm số đường / tổng lớn nhất** dọc theo các đường như vậy. → DP trên DAG: `f(ô) = kết hợp f(các ô kế tiếp)`.
- Kết quả tại một ô **không phụ thuộc đường đã đi tới nó**. → Memo được. (Ngược lại, nếu phải nhớ "đã đi qua những ô nào" thì thường là backtracking, không memo được.)

Ba cách cài cho cùng một ý tưởng: **DFS + memo** (ngắn nhất), **Kahn bóc lớp** (không đệ quy, số lớp = đường dài nhất), **sắp xếp theo giá trị rồi DP** (thứ tự topo có sẵn nhờ giá trị).

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 2328. Number of Increasing Paths in a Grid (Hard) | cùng đồ thị, nhưng **đếm** số đường tăng (thay `max` bằng `sum`) — Bài tập 5 |
| 2713. Maximum Strictly Increasing Cells in a Matrix (Hard) | được nhảy tới mọi ô cùng hàng/cùng cột; giải bằng sắp xếp theo giá trị + DP theo hàng/cột (họ hàng của Cách 4) |
| 300. Longest Increasing Subsequence (Medium) | phiên bản 1 chiều; `dp[i]` = dãy tăng dài nhất **kết thúc** tại `i`, giống tư duy Cách 4 |
| 207. Course Schedule / 210. Course Schedule II (Medium) | thuật toán Kahn với bậc vào; phát hiện chu trình |
| 1136. Parallel Courses (Medium, Premium) | số học kỳ tối thiểu = số lớp Kahn = chuỗi môn tiên quyết dài nhất — đúng ý tưởng của Cách 3 |
| 1857. Largest Color Value in a Directed Graph (Hard) | DP trên DAG theo thứ tự Kahn |
| 417. Pacific Atlantic Water Flow (Medium) | DFS trên lưới theo độ cao (nhưng không ngặt → cần `visited`) |

## 15. Bài tập vận dụng

Làm theo thứ tự, tự làm xong rồi mới xem **Gợi ý** (mục 16) và **Đáp án** (mục 17).

### Bài 1 — Đoán output (không chạy code)

1. `lip_memo([[1, 2], [4, 3]])` trả về bao nhiêu?
2. Bảng memo (đường dài nhất **bắt đầu** tại mỗi ô) của `[[1, 2], [4, 3]]` trông thế nào?
3. `lip_memo([[7, 7], [7, 7]])` trả về bao nhiêu?
4. `lip_topo([[3, 3, 3], [3, 0, 3], [3, 3, 3]])` bóc bao nhiêu lớp? Lớp 1 có bao nhiêu ô?
5. Với `[[1, 5, 2]]`, bản đúng trả `2`. Nếu sửa điều kiện thành `!=` (khác nhau là được đi) thì chuyện gì xảy ra?

### Bài 2 — Tính tay bằng bảng

Với ma trận

```text
M = [[5, 3, 6],
     [4, 1, 2],
     [9, 8, 7]]
```

a) Tính tay bảng memo `f(r, c)` (đường tăng dài nhất bắt đầu tại mỗi ô). Gợi ý: tính các "đỉnh núi" trước.

b) Chạy tay thuật toán Kahn: liệt kê các ô của từng lớp. Kiểm tra lại: lớp của mỗi ô có trùng với `f` không? Đáp án là bao nhiêu?

### Bài 3 — Điền TODO: DFS + memo

Viết lại lời giải chính mà **không nhìn** mục 5. Các TODO dẫn đường từng bước.

In [28]:
def lip_exercise(matrix):
    m, n = len(matrix), len(matrix[0])
    # TODO 1: tạo bảng memo m x n toàn 0 (cẩn thận bẫy [[0] * n] * m)

    def dfs(r, c):
        # TODO 2: nếu memo[r][c] đã khác 0 thì trả luôn
        # TODO 3: best = 1; duyệt 4 hướng trong DIRS
        # TODO 4: nếu (nr, nc) trong lưới VÀ matrix[nr][nc] > matrix[r][c]:
        #             best = max(best, 1 + dfs(nr, nc))
        # TODO 5: ghi memo[r][c] = best rồi trả best
        pass

    # TODO 6: trả về max của dfs(r, c) trên mọi ô
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert lip_exercise(ex1) == 4
# assert lip_exercise(ex2) == 4
# assert lip_exercise(ex3) == 1
# assert lip_exercise([[7, 7], [7, 7]]) == 1
# assert lip_exercise([[1, 5, 2]]) == 2

### Bài 4 — Tự viết: Kahn theo chiều ngược (bắt đầu từ "đáy thung lũng")

Mục 8 bóc từ **đỉnh núi** (bậc ra 0). Hãy viết `lip_topo_from_valleys(matrix)` bóc từ **đáy thung lũng**: các ô có **bậc vào 0** (không có hàng xóm nào **nhỏ hơn**). Khi bóc một ô, giảm bậc vào của các hàng xóm **lớn hơn** nó. Số lớp vẫn phải bằng đáp án.

Câu hỏi phụ: với chiều này, "lớp của một ô" bằng bảng nào — memo của Cách 2 hay `dp` của Cách 4?

In [29]:
def lip_topo_from_valleys(matrix):
    # TODO 1: tính in_deg[r][c] = số hàng xóm NHỎ hơn (r, c)
    # TODO 2: đưa mọi ô có in_deg == 0 vào deque
    # TODO 3: while q: layers += 1; bóc đúng len(q) ô;
    #         với mỗi hàng xóm LỚN hơn: in_deg -= 1, về 0 thì append
    # TODO 4: trả về layers
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert lip_topo_from_valleys(ex1) == 4
# assert lip_topo_from_valleys(ex2) == 4
# assert lip_topo_from_valleys(ex3) == 1
# assert lip_topo_from_valleys([[5, 3, 6], [4, 1, 2], [9, 8, 7]]) == 5

### Bài 5 — Biến thể: LeetCode 2328. Number of Increasing Paths in a Grid

Cho lưới `grid`, đếm **số đường tăng ngặt** (đi 4 hướng, độ dài từ 1 trở lên; hai đường khác nhau nếu dãy ô khác nhau). Đáp án có thể rất lớn nên trả về theo **modulo `10^9 + 7`**.

```text
grid = [[1, 1], [3, 4]]  ->  8
    độ dài 1: [1], [1], [3], [4]          (4 đường)
    độ dài 2: [1→3], [1→4], [3→4]         (3 đường)
    độ dài 3: [1→3→4]                     (1 đường)
grid = [[1], [2]]        ->  3            ([1], [2], [1→2])
```

Gợi ý tư duy: đặt `g(r, c)` = số đường tăng **bắt đầu** tại `(r, c)`. Công thức chỉ khác bài 329 ở một chữ.

In [30]:
MOD = 10**9 + 7


def count_increasing_paths(grid):
    # TODO: g(r, c) = 1 + tổng g(hàng xóm lớn hơn)   (mod MOD)
    #       đáp án = tổng g(r, c) trên mọi ô       (mod MOD)
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert count_increasing_paths([[1, 1], [3, 4]]) == 8
# assert count_increasing_paths([[1], [2]]) == 3
# assert count_increasing_paths([[7]]) == 1

### Bài 6 — Mini challenge: trả về **chính đường đi**, ưu tiên dãy nhỏ nhất theo thứ tự từ điển

Viết `lexi_smallest_longest_path(matrix)` trả về **danh sách giá trị** của một đường tăng dài nhất. Nếu có nhiều đường cùng dài nhất, chọn đường có dãy giá trị **nhỏ nhất theo thứ tự từ điển** (lexicographic — so phần tử đầu, bằng nhau mới so phần tử tiếp theo, giống so sánh tuple ở mục 9.1; Python so được trực tiếp: `[2, 4, 5, 6] < [3, 4, 5, 6]` là `True`).

```text
ex1 -> [1, 2, 6, 9]
ex2 -> [2, 4, 5, 6]      (ex2 có hai đường dài 4: [3,4,5,6] và [2,4,5,6]; đường sau nhỏ hơn)
```

Thử thách: chứng minh (bằng lời) vì sao chọn "đuôi" tốt nhất ở từng ô rồi ghép lại vẫn cho kết quả nhỏ nhất toàn cục.

In [31]:
def lexi_smallest_longest_path(matrix):
    # TODO: memo cho mỗi ô một tuple = đường tốt nhất BẮT ĐẦU tại ô đó
    #       (dài nhất; cùng dài thì nhỏ nhất theo thứ tự từ điển)
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert lexi_smallest_longest_path(ex1) == [1, 2, 6, 9]
# assert lexi_smallest_longest_path(ex2) == [2, 4, 5, 6]
# assert lexi_smallest_longest_path(ex3) == [1]

## 16. Gợi ý theo mức

**Bài 1**
- Mức 1: vẽ mũi tên sang ô lớn hơn cho từng ô.
- Mức 2: ở câu 4, ô nào không có hàng xóm lớn hơn? Ô `3` có hàng xóm `3` (bằng) — có tính là lớn hơn không?
- Mức 3: ở câu 5, `!=` cho phép đi cả sang ô **nhỏ hơn** → giữa hai ô khác nhau có mũi tên **hai chiều**.

**Bài 2**
- Mức 1: đỉnh núi của `M` là các ô không có hàng xóm lớn hơn: tìm được 3 ô.
- Mức 2: ô trung tâm `1` có 4 hàng xóm `3, 8, 4, 2` — đều lớn hơn nó.
- Mức 3: `f(1) = 1 + max(f(3), f(8), f(4), f(2))`; tính `f(2)` qua `7 → 8 → 9`.

**Bài 3**
- Mức 1: chép lại khung Cách 1, thêm đúng 2 dòng memo.
- Mức 2: `memo = [[0] * n for _ in range(m)]`; dòng đầu của `dfs` là `if memo[r][c]: return memo[r][c]`.
- Mức 3: `return max(dfs(r, c) for r in range(m) for c in range(n))`.

**Bài 4**
- Mức 1: chỉ đảo hai chỗ so sánh của `lip_topo`: `>` ↔ `<`.
- Mức 2: `in_deg[r][c] += 1` khi hàng xóm **nhỏ hơn**; khi bóc `(r, c)`, giảm `in_deg` của hàng xóm **lớn hơn**.
- Mức 3: lớp của ô = đường dài nhất **kết thúc** tại ô → trùng `dp` của Cách 4.

**Bài 5**
- Mức 1: "dài nhất" dùng `max`, "đếm" dùng `sum`.
- Mức 2: `g(r, c) = 1 + sum(g(hàng xóm lớn hơn))` — số `1` là đường chỉ gồm chính ô đó.
- Mức 3: lấy `% MOD` sau mỗi phép cộng; vẫn memo được vì `g` chỉ phụ thuộc `(r, c)`. Có thể dùng Cách 4 (sắp giảm dần) để tránh đệ quy.

**Bài 6**
- Mức 1: memo không lưu số nữa mà lưu **tuple** giá trị của đường tốt nhất bắt đầu tại ô.
- Mức 2: ở ô `(r, c)`, chọn "đuôi" tốt nhất trong các hàng xóm lớn hơn: dài hơn thì thắng; cùng dài thì tuple nhỏ hơn thắng. Kết quả là `(matrix[r][c],) + đuôi`.
- Mức 3: ở vòng ngoài cũng so sánh theo đúng quy tắc đó giữa các ô xuất phát.

## 17. Đáp án tham khảo

### Đáp án Bài 1

1. `4`: đường `1 → 2 → 3 → 4` đi hình chữ U: `(0,0)=1 → (0,1)=2 → (1,1)=3 → (1,0)=4`.
2. Bảng memo:
   ```text
     4  3
     1  2
   ```
   Ô `4` là đỉnh núi (`1`); ô `3` chỉ lên được `4` (`2`); ô `2` → `3` (`3`); ô `1` → `2` (`4`). Ô `1` cũng kề ô `4` nhưng đi thẳng sang đó chỉ được `2`.
3. `1`: không có bước nào tăng ngặt.
4. **2 lớp**. Lớp 1 có **8 ô**: mọi ô `3` đều là đỉnh núi (hàng xóm của chúng là `3` bằng nhau hoặc `0` nhỏ hơn). Ô `0` ở giữa có bậc ra 4, bị giảm 4 lần rồi vào lớp 2.
5. Có mũi tên hai chiều `1 ⇄ 5`, nên DFS gọi qua gọi lại mãi → `RecursionError` (giống Lỗi 1). Điều kiện "tăng ngặt" là thứ giữ cho đồ thị không có chu trình.

In [32]:
ans, memo_b1 = lip_memo([[1, 2], [4, 3]], return_memo=True)
print("1.", ans)
print_grid(memo_b1, "2. bảng memo:")
print("3.", lip_memo([[7, 7], [7, 7]]))
layers, layer_of = lip_topo([[3, 3, 3], [3, 0, 3], [3, 3, 3]], return_layers=True)
print("4. số lớp =", layers, "| số ô ở lớp 1 =", sum(row.count(1) for row in layer_of))
assert ans == 4 and memo_b1 == [[4, 3], [1, 2]] and layers == 2


def lip_wrong_neq(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]

    def dfs(r, c):
        if memo[r][c]:
            return memo[r][c]
        best = 1
        for nr, nc in neighbors(matrix, r, c):
            if matrix[nr][nc] != matrix[r][c]:          # SAI: khác nhau là được đi
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n))


old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    lip_wrong_neq([[1, 5, 2]])
except RecursionError as e:
    print("5. điều kiện != -> RecursionError:", e)
finally:
    sys.setrecursionlimit(old_limit)

1. 4
2. bảng memo:
  4   3
  1   2
3. 1
4. số lớp = 2 | số ô ở lớp 1 = 8
5. điều kiện != -> RecursionError: maximum recursion depth exceeded


### Đáp án Bài 2

a) Bảng `f` (đỉnh núi `5`, `6`, `9` có `f = 1`):

```text
M:            f:
 5  3  6       1  2  1
 4  1  2       2  5  4
 9  8  7       1  2  3
```

Tính dần: `f(8) = 1 + f(9) = 2`; `f(7) = 1 + f(8) = 3`; `f(2) = 1 + max(f(6), f(7)) = 1 + 3 = 4`; `f(3) = 1 + max(f(5), f(6)) = 2`; `f(4) = 1 + max(f(5), f(9)) = 2`; `f(1) = 1 + max(f(3), f(8), f(4), f(2)) = 1 + 4 = 5`.

b) Các lớp Kahn:

| Lớp | Các ô |
|:---:|---|
| 1 | (0,0)=5, (0,2)=6, (2,0)=9 |
| 2 | (0,1)=3, (1,0)=4, (2,1)=8 |
| 3 | (2,2)=7 |
| 4 | (1,2)=2 |
| 5 | (1,1)=1 |

Lớp của mỗi ô trùng đúng bảng `f`. Đáp án **5**: đường `1 → 2 → 7 → 8 → 9`.

In [33]:
M = [[5, 3, 6], [4, 1, 2], [9, 8, 7]]
ans, memo_M = lip_memo(M, return_memo=True)
print_grid(memo_M, "bảng f:")
assert memo_M == [[1, 2, 1], [2, 5, 4], [1, 2, 3]] and ans == 5
trace_lip_topo(M)

bảng f:
  1   2   1
  2   5   4
  1   2   3
Lớp 1: bóc (0,0)=5, (0,2)=6, (2,0)=9
   giảm bậc ra: (1,0):2->1; (0,1):2->1; (1,2):2->1; (0,1):1->0; (1,0):1->0; (2,1):1->0
   vào lớp sau: ['(0,1)', '(1,0)', '(2,1)']
Lớp 2: bóc (0,1)=3, (1,0)=4, (2,1)=8
   giảm bậc ra: (1,1):4->3; (1,1):3->2; (1,1):2->1; (2,2):1->0
   vào lớp sau: ['(2,2)']
Lớp 3: bóc (2,2)=7
   giảm bậc ra: (1,2):1->0
   vào lớp sau: ['(1,2)']
Lớp 4: bóc (1,2)=2
   giảm bậc ra: (1,1):1->0
   vào lớp sau: ['(1,1)']
Lớp 5: bóc (1,1)=1
   giảm bậc ra: (không có)
   vào lớp sau: (hết)
Số lớp = 5


5

### Đáp án Bài 3 và Bài 4

Bài 3 chính là `lip_memo`. Bài 4 đảo chiều so sánh của `lip_topo`; lớp của một ô lúc này là đường tăng dài nhất **kết thúc** tại ô đó — trùng bảng `dp` của Cách 4 (cell dưới kiểm chứng trên Example 1).

In [34]:
def lip_exercise_solution(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = [[0] * n for _ in range(m)]

    def dfs(r, c):
        if memo[r][c]:
            return memo[r][c]
        best = 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        memo[r][c] = best
        return best

    return max(dfs(r, c) for r in range(m) for c in range(n))


def lip_topo_from_valleys_solution(matrix, return_layers=False):
    m, n = len(matrix), len(matrix[0])
    in_deg = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            for nr, nc in neighbors(matrix, r, c):
                if matrix[nr][nc] < matrix[r][c]:          # mũi tên (nr,nc) -> (r,c)
                    in_deg[r][c] += 1
    q = deque((r, c) for r in range(m) for c in range(n) if in_deg[r][c] == 0)
    layer_of = [[0] * n for _ in range(m)]
    layers = 0
    while q:
        layers += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            layer_of[r][c] = layers
            for nr, nc in neighbors(matrix, r, c):
                if matrix[nr][nc] > matrix[r][c]:
                    in_deg[nr][nc] -= 1
                    if in_deg[nr][nc] == 0:
                        q.append((nr, nc))
    return (layers, layer_of) if return_layers else layers


for f in (lip_exercise_solution, lip_topo_from_valleys_solution):
    assert f(ex1) == 4 and f(ex2) == 4 and f(ex3) == 1
    assert f([[7, 7], [7, 7]]) == 1 and f([[1, 5, 2]]) == 2
    assert f([[5, 3, 6], [4, 1, 2], [9, 8, 7]]) == 5
_, layer_of = lip_topo_from_valleys_solution(ex1, return_layers=True)
print_grid(layer_of, "Bài 4 - lớp của từng ô (Example 1) = dp 'kết thúc tại' của Cách 4:")

random.seed(3290)
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    mat = [[random.randint(0, 4) for _ in range(n)] for _ in range(m)]
    assert lip_exercise_solution(mat) == lip_topo_from_valleys_solution(mat) == lip_bruteforce(mat)
print("Bài 3 & Bài 4 PASS (kể cả 300 lưới ngẫu nhiên)")

Bài 4 - lớp của từng ô (Example 1) = dp 'kết thúc tại' của Cách 4:
  4   3   1
  3   2   3
  2   1   1
Bài 3 & Bài 4 PASS (kể cả 300 lưới ngẫu nhiên)


### Đáp án Bài 5 — Number of Increasing Paths in a Grid

`g(r, c) = 1 + Σ g(hàng xóm lớn hơn)`: số `1` là đường chỉ gồm `(r, c)`; mỗi đường dài hơn bắt đầu bằng `(r, c)` rồi đi tiếp sang đúng một hàng xóm lớn hơn, và các nhóm đường theo từng hàng xóm là rời nhau, nên **cộng** lại. Đáp án = `Σ g` trên mọi ô.

Bản dưới dùng **Cách 4 với thứ tự giảm dần** (ô lớn tính trước) để khỏi lo đệ quy. Kiểm tra với brute force liệt kê từng đường.

In [35]:
def count_increasing_paths_solution(grid):
    m, n = len(grid), len(grid[0])
    g = [[1] * n for _ in range(m)]             # g[r][c] = số đường tăng BẮT ĐẦU tại (r, c)
    for v, r, c in sorted(((grid[r][c], r, c) for r in range(m) for c in range(n)), reverse=True):
        for nr, nc in neighbors(grid, r, c):
            if grid[nr][nc] > v:                 # hàng xóm lớn hơn đã tính xong
                g[r][c] = (g[r][c] + g[nr][nc]) % MOD
    return sum(map(sum, g)) % MOD


def count_paths_brute(grid):
    total = 0

    def dfs(r, c):
        nonlocal total
        total += 1                               # đường kết thúc tại (r, c)
        for nr, nc in neighbors(grid, r, c):
            if grid[nr][nc] > grid[r][c]:
                dfs(nr, nc)

    for r in range(len(grid)):
        for c in range(len(grid[0])):
            dfs(r, c)
    return total


assert count_increasing_paths_solution([[1, 1], [3, 4]]) == 8
assert count_increasing_paths_solution([[1], [2]]) == 3
assert count_increasing_paths_solution([[7]]) == 1
random.seed(2328)
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    grid_t = [[random.randint(0, 5) for _ in range(n)] for _ in range(m)]
    assert count_increasing_paths_solution(grid_t) == count_paths_brute(grid_t)
print("ex1 của bài 329 có", count_increasing_paths_solution(ex1), "đường tăng")
print("Bài 5 PASS")

ex1 của bài 329 có 23 đường tăng
Bài 5 PASS


### Đáp án Bài 6 — Đường dài nhất, nhỏ nhất theo thứ tự từ điển

Memo của mỗi ô là tuple `best(r, c)` = đường tốt nhất bắt đầu tại ô đó. Ở ô `(r, c)`, mọi đường bắt đầu tại đây đều có dạng `(matrix[r][c],) + đuôi`, với "đuôi" là một đường bắt đầu tại hàng xóm lớn hơn. Phần tử đầu đã cố định, nên đường dài nhất & nhỏ nhất theo từ điển phải có **đuôi dài nhất & nhỏ nhất theo từ điển** — và với mỗi hàng xóm, đuôi tốt nhất chính là `best(hàng xóm)` (giả thiết quy nạp). Vậy chỉ cần so các `best(hàng xóm)`.

Bộ nhớ: mỗi ô lưu cả một tuple, tệ nhất `O(N · L)` (`L` = độ dài đường). Đủ dùng cho bài luyện tập; muốn tiết kiệm có thể chỉ lưu "ô kế tiếp" rồi so sánh bằng cách đi dọc hai đường.

In [36]:
def better(a, b):
    # a tốt hơn b nếu dài hơn, hoặc cùng dài mà nhỏ hơn theo thứ tự từ điển
    return len(a) > len(b) or (len(a) == len(b) and a < b)


def lexi_smallest_longest_path_solution(matrix):
    m, n = len(matrix), len(matrix[0])
    memo = {}

    def best(r, c):
        if (r, c) in memo:
            return memo[(r, c)]
        tail = ()
        for nr, nc in neighbors(matrix, r, c):
            if matrix[nr][nc] > matrix[r][c]:
                cand = best(nr, nc)
                if better(cand, tail):
                    tail = cand
        memo[(r, c)] = (matrix[r][c],) + tail
        return memo[(r, c)]

    answer = ()
    for r in range(m):
        for c in range(n):
            if better(best(r, c), answer):
                answer = best(r, c)
    return list(answer)


def lexi_brute(matrix):
    answer = []
    for r in range(len(matrix)):
        for c in range(len(matrix[0])):
            for p in all_maximal_paths(matrix, r, c):
                vals = [matrix[i][j] for i, j in p]
                if len(vals) > len(answer) or (len(vals) == len(answer) and vals < answer):
                    answer = vals
    return answer


assert lexi_smallest_longest_path_solution(ex1) == [1, 2, 6, 9]
assert lexi_smallest_longest_path_solution(ex2) == [2, 4, 5, 6]
assert lexi_smallest_longest_path_solution(ex3) == [1]
random.seed(6)
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    mat = [[random.randint(0, 6) for _ in range(n)] for _ in range(m)]
    assert lexi_smallest_longest_path_solution(mat) == lexi_brute(mat), mat
print("M của Bài 2:", lexi_smallest_longest_path_solution([[5, 3, 6], [4, 1, 2], [9, 8, 7]]))
print("Bài 6 PASS")

M của Bài 2: [1, 2, 7, 8, 9]
Bài 6 PASS


## 18. Template cuối cùng nên nhớ

```python
class Solution:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:
        sys.setrecursionlimit(100000)
        m, n = len(matrix), len(matrix[0])
        memo = [[0] * n for _ in range(m)]          # 0 = chưa tính
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))

        def dfs(r, c):
            if memo[r][c]:
                return memo[r][c]
            best = 1                                  # đếm số Ô, ô đơn lẻ = 1
            for dr, dc in dirs:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and matrix[nr][nc] > matrix[r][c]:   # tăng NGẶT
                    best = max(best, 1 + dfs(nr, nc))
            memo[r][c] = best                         # ghi sổ
            return best

        return max(dfs(r, c) for r in range(m) for c in range(n))   # mọi ô xuất phát
```

Bản không đệ quy (Kahn): *tính bậc ra → đưa đỉnh núi vào `deque` → bóc từng lớp, giảm bậc ra của hàng xóm nhỏ hơn → trả số lớp.*

**Câu thần chú**: *"Tăng ngặt nên không cần visited. Không cần visited nên memo được. Mỗi ô tính một lần: `1 + max` của hàng xóm lớn hơn. Lấy max trên mọi ô."*

## 19. Checklist tự kiểm tra trước khi submit

- Điều kiện đi là `>` (tăng **ngặt**), **không** phải `>=` hay `!=`, đúng chưa?
- Kiểm tra biên đủ **cả hai đầu**: `0 <= nr < m and 0 <= nc < n` (đừng để chỉ số âm "quấn vòng")?
- Bảng memo tạo bằng `[[0] * n for _ in range(m)]`, **không** phải `[[0] * n] * m`?
- Có ghi `memo[r][c] = best` trước khi `return` chưa (nếu không sẽ thành hàm mũ)?
- Base case là `best = 1` (đếm **ô**), không phải `0`?
- Đáp án lấy `max` trên **mọi** ô xuất phát, không chỉ `(0, 0)`?
- Nếu dùng `@lru_cache`: hàm `dfs` nằm **bên trong** method (không dính cache giữa các test)?
- Đã nghĩ tới đường tăng cực dài (tới `m · n` ô): nới `sys.setrecursionlimit` cho bản bảng memo, hoặc dùng bản Kahn / sort không đệ quy?
- Đã test: 3 ví dụ của đề, lưới toàn bằng nhau, một hàng, một cột, `[[1, 5, 2]]` (bẫy quấn vòng), và một lưới `200 × 200` chưa?